<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03_feature_engineering**

# **0. Configuración del Entorno**


## 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [3]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [4]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

## 0.4. Definición de rutas



In [5]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [6]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/03_targets/mnq_intraday_t2_ti.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [7]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## 0.5. Códigos auxiliares para carga de datos y visualización


In [8]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [9]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

In [13]:
mnq_intraday_t2_ti = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday_t2_ti, name="mnq_intraday_t2_ti", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday_t2_ti
Shape: (779590, 66)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60', 'close_fwd_30', 'delta_30', 't1_dir_bin_30', 'close_fwd_60', 'delta_60', 't1_dir_bin_60', 'close_fwd_90', 'delta_90', 't1_dir_bin_90', 'close_fwd_120', 'delta_120', 't1_dir_bin_120', 't2_dir_thr_30', 't2_dir_thr_60', 't2_dir_thr_90', 't2_dir_thr_120']
Index: DatetimeIndex | TZ: America/New

## 0.6. Auxiliares

In [11]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [12]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

# **1. Análisis del dataset de entrada y targets**

In [18]:
import numpy as np
import pandas as pd


def analyze_input_dataset_t2_ti(
    df: pd.DataFrame,
    *,
    datetime_index_required: bool = True,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    regime_col: str = "regime_id",
    ohlcv_cols: tuple[str, ...] = ("open", "high", "low", "close", "volume"),
    ti_cols: tuple[str, ...] = (
        "rsi_14", "rsi_7", "rsi_5", "rsi_3",
        "mom_10", "mom_5", "mom_3",
        "volume_ratio_15", "volume_ratio_20", "volume_ratio_30",
        "volume_ratio_60", "volume_ratio_90",
        "macd",
        "ema_15", "ema_20", "ema_30", "ema_60",
        "stoch_k_14", "stoch_k_20", "stoch_k_30",
        "bb_15_15", "bb_20_15", "bb_30_15", "bb_60_15",
        "bb_15_20", "bb_20_20", "bb_30_20", "bb_60_20",
        "bb_15_25", "bb_20_25", "bb_30_25", "bb_60_25",
        "atr_norm_5", "atr_norm_10", "atr_norm_14", "atr_norm_20", "atr_norm_30",
        "roc_5", "roc_10", "roc_20", "roc_30", "roc_60",
    ),
    target_cols: tuple[str, ...] = (
        "t2_dir_thr_30",
        "t2_dir_thr_60",
        "t2_dir_thr_90",
        "t2_dir_thr_120",
    ),
    print_report: bool = True,
) -> dict:
    """
    Analiza el dataset mnq_intraday_t2_ti antes de modelado.

    Verifica:
    - presencia de columnas esperadas
    - orden temporal correcto
    - consistencia básica OHLCV
    - consistencia de regime_id
    - ausencia básica de leakage temporal en targets T2
    - distribución multiclase de los targets
    - NaNs en indicadores técnicos
    - duplicados temporales

    Retorna un diccionario con:
    - report
    - target_summary
    - regime_summary
    - ti_nan_summary
    - invalid_ohlc_rows
    - invalid_regime_rows
    - leakage_rows
    """

    data = df.copy()

    # ============================================================
    # 0. Validación de columnas
    # ============================================================
    required_cols = set(ohlcv_cols) | set(ti_cols) | set(target_cols) | {date_col, minute_col, regime_col}
    missing_cols = sorted([c for c in required_cols if c not in data.columns])

    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    # ============================================================
    # 1. Validación / normalización temporal
    # ============================================================
    if datetime_index_required and not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener DatetimeIndex.")

    data[date_col] = pd.to_datetime(data[date_col])

    if isinstance(data.index, pd.DatetimeIndex):
        data = data.sort_index().copy()
        is_sorted = data.index.is_monotonic_increasing
    else:
        data = data.sort_values([date_col, minute_col]).copy()
        is_sorted = (
            data[[date_col, minute_col]]
            .reset_index(drop=True)
            .equals(
                data[[date_col, minute_col]]
                .sort_values([date_col, minute_col])
                .reset_index(drop=True)
            )
        )

    # ============================================================
    # 2. Consistencia OHLCV
    # ============================================================
    invalid_ohlc_mask = (
        (data["high"] < data["low"]) |
        (data["open"] > data["high"]) |
        (data["open"] < data["low"]) |
        (data["close"] > data["high"]) |
        (data["close"] < data["low"]) |
        (data["volume"] < 0)
    )
    invalid_ohlc_rows = data.loc[invalid_ohlc_mask].copy()

    # ============================================================
    # 3. Consistencia de regime_id
    # ============================================================
    invalid_regime_mask = data[regime_col].isna()
    invalid_regime_rows = data.loc[invalid_regime_mask].copy()

    regime_summary = (
        data[regime_col]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis(regime_col)
        .reset_index(name="n_rows")
    )

    # ============================================================
    # 4. Leakage temporal básico
    #    Si no hay futuro suficiente dentro del día, el target debería ser NaN
    # ============================================================
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("Para validar leakage temporal, el DataFrame debe tener DatetimeIndex.")

    leakage_checks = []

    last_ts_by_day = data.groupby(date_col).apply(lambda g: g.index.max())
    last_ts_by_day.name = "last_ts_day"

    data = data.merge(
        last_ts_by_day,
        left_on=date_col,
        right_index=True,
        how="left"
    )

    for target_col in target_cols:
        horizon = int(target_col.split("_")[-1])
        expected_future_ts = data.index + pd.Timedelta(minutes=horizon)

        invalid_future_mask = expected_future_ts > data["last_ts_day"]
        suspect_non_nan_target_mask = invalid_future_mask & data[target_col].notna()

        leakage_checks.append(
            data.loc[suspect_non_nan_target_mask, [date_col, minute_col, target_col]].assign(
                target_col_name=target_col
            )
        )

    leakage_rows = pd.concat(leakage_checks, axis=0) if leakage_checks else pd.DataFrame()

    # ============================================================
    # 5. Resumen de targets T2 (multiclase)
    # ============================================================
    target_summary_rows = []

    for target_col in target_cols:
        s = data[target_col]

        valid = s.dropna()
        n_total = len(s)
        n_notna = valid.shape[0]
        n_nan = s.isna().sum()

        n_neg1 = (valid == -1).sum()
        n_0 = (valid == 0).sum()
        n_pos1 = (valid == 1).sum()

        target_summary_rows.append({
            "target_col": target_col,
            "n_total": int(n_total),
            "n_notna": int(n_notna),
            "n_nan": int(n_nan),
            "pct_nan": float(n_nan / n_total),
            "n_-1": int(n_neg1),
            "n_0": int(n_0),
            "n_1": int(n_pos1),
            "pct_-1": float(n_neg1 / n_notna) if n_notna > 0 else np.nan,
            "pct_0": float(n_0 / n_notna) if n_notna > 0 else np.nan,
            "pct_1": float(n_pos1 / n_notna) if n_notna > 0 else np.nan,
            "majority_class": valid.value_counts().idxmax() if n_notna > 0 else np.nan,
        })

    target_summary = pd.DataFrame(target_summary_rows)

    # ============================================================
    # 6. NaNs en indicadores técnicos
    # ============================================================
    ti_nan_summary = pd.DataFrame({
        "feature": list(ti_cols),
        "n_nan": [int(data[c].isna().sum()) for c in ti_cols],
        "pct_nan": [float(data[c].isna().mean()) for c in ti_cols],
    }).sort_values(["pct_nan", "feature"], ascending=[False, True])

    # ============================================================
    # 7. Duplicados temporales
    # ============================================================
    if isinstance(data.index, pd.DatetimeIndex):
        dup_time_count = int(data.index.duplicated().sum())
    else:
        dup_time_count = int(data.duplicated(subset=[date_col, minute_col]).sum())

    # ============================================================
    # 8. Resumen general
    # ============================================================
    report = {
        "n_rows": int(len(data)),
        "n_cols": int(data.shape[1]),
        "datetime_index_type": str(type(data.index)),
        "is_sorted_temporally": bool(is_sorted),
        "n_unique_days": int(data[date_col].nunique()),
        "datetime_duplicates": dup_time_count,
        "n_invalid_ohlc_rows": int(len(invalid_ohlc_rows)),
        "n_invalid_regime_rows": int(len(invalid_regime_rows)),
        "n_suspect_leakage_rows": int(len(leakage_rows)),
        "n_ti_features": int(len(ti_cols)),
    }

    # ============================================================
    # 9. Print reporte
    # ============================================================
    if print_report:
        print("=" * 100)
        print("ANÁLISIS DEL DATASET | mnq_intraday_t2_ti")
        print("=" * 100)
        print(f"Filas: {report['n_rows']:,}")
        print(f"Columnas: {report['n_cols']}")
        print(f"Días únicos: {report['n_unique_days']}")
        print(f"Orden temporal correcto: {report['is_sorted_temporally']}")
        print(f"Timestamps duplicados: {report['datetime_duplicates']}")
        print(f"Filas con OHLCV inconsistente: {report['n_invalid_ohlc_rows']}")
        print(f"Filas con regime_id inválido: {report['n_invalid_regime_rows']}")
        print(f"Filas sospechosas de leakage temporal: {report['n_suspect_leakage_rows']}")
        print(f"Número de indicadores técnicos: {report['n_ti_features']}")
        print()

        print("-" * 100)
        print("MUESTRAS POR RÉGIMEN")
        print("-" * 100)
        print(regime_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("RESUMEN DE TARGETS T2")
        print("-" * 100)
        print(target_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("TOP 15 FEATURES CON MÁS NaNs")
        print("-" * 100)
        print(ti_nan_summary.head(15).to_string(index=False))
        print()

        if len(invalid_ohlc_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS CON OHLCV INVÁLIDO")
            print("-" * 100)
            print(invalid_ohlc_rows.head(10).to_string())

        if len(invalid_regime_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS CON REGIME_ID INVÁLIDO")
            print("-" * 100)
            print(invalid_regime_rows.head(10).to_string())

        if len(leakage_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS SOSPECHOSAS DE LEAKAGE")
            print("-" * 100)
            print(leakage_rows.head(10).to_string(index=False))

    # limpieza
    if "last_ts_day" in data.columns:
        data = data.drop(columns=["last_ts_day"])

    return {
        "report": report,
        "target_summary": target_summary,
        "regime_summary": regime_summary,
        "ti_nan_summary": ti_nan_summary,
        "invalid_ohlc_rows": invalid_ohlc_rows,
        "invalid_regime_rows": invalid_regime_rows,
        "leakage_rows": leakage_rows,
    }

In [19]:
time_cols = ["date", "minute_of_day"]
regimen_cols = ["regime_id"]

ohlcv_cols = ["open", "high", "low", "close", "volume"]

ti_cols = [
    "rsi_14", "rsi_7", "rsi_5", "rsi_3", "mom_10", "mom_5", "mom_3",
    "volume_ratio_15", "volume_ratio_20", "volume_ratio_30",
    "volume_ratio_60", "volume_ratio_90", "macd", "ema_15", "ema_20",
    "ema_30", "ema_60", "stoch_k_14", "stoch_k_20", "stoch_k_30",
    "bb_15_15", "bb_20_15", "bb_30_15", "bb_60_15", "bb_15_20", "bb_20_20",
    "bb_30_20", "bb_60_20", "bb_15_25", "bb_20_25", "bb_30_25", "bb_60_25",
    "atr_norm_5", "atr_norm_10", "atr_norm_14", "atr_norm_20", "atr_norm_30",
    "roc_5", "roc_10", "roc_20", "roc_30", "roc_60",
]

targets_cols = [
    "t2_dir_thr_30",
    "t2_dir_thr_60",
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

dataset_report = analyze_input_dataset_t2_ti(
    mnq_intraday_t2_ti,
    date_col="date",
    minute_col="minute_of_day",
    regime_col="regime_id",
    ohlcv_cols=tuple(ohlcv_cols),
    ti_cols=tuple(ti_cols),
    target_cols=tuple(targets_cols),
    print_report=True,
)

ANÁLISIS DEL DATASET | mnq_intraday_t2_ti
Filas: 779,590
Columnas: 67
Días únicos: 1295
Orden temporal correcto: True
Timestamps duplicados: 0
Filas con OHLCV inconsistente: 0
Filas con regime_id inválido: 0
Filas sospechosas de leakage temporal: 0
Número de indicadores técnicos: 42

----------------------------------------------------------------------------------------------------
MUESTRAS POR RÉGIMEN
----------------------------------------------------------------------------------------------------
 regime_id  n_rows
         0  196840
         1   77700
         2   77700
         3  388500
         4   38850

----------------------------------------------------------------------------------------------------
RESUMEN DE TARGETS T2
----------------------------------------------------------------------------------------------------
    target_col  n_total  n_notna  n_nan  pct_nan   n_-1    n_0    n_1   pct_-1    pct_0    pct_1  majority_class
 t2_dir_thr_30   779590   740740  38850 

# **3. Separación IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:

* el cálculo y selección de indicadores técnicos,
* el análisis del Information Coefficient (IC),
* la evaluación de dependencias y colinealidad entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:

* validar la estabilidad temporal de las relaciones observadas,
* comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
* verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.
Los indicadores se seleccionan en base a su desempeño en el conjunto IS y se validan posteriormente en OOS.
Un indicador solo se considera robusto si mantiene un comportamiento consistente fuera de muestra, tanto en el signo como en la estabilidad de la magnitud del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.

**Criterio propuesto (ajustable)**

- IS: desde 2019-12-23 hasta 2022-12-31
- OOS: desde 2023-01-01 hasta 2025-06-13

Este split es solo para selección y validación de features, no es el split final de modelado.

In [21]:
import numpy as np
import pandas as pd

def add_is_oos_split(
    df: pd.DataFrame,
    *,
    cut_date: str | pd.Timestamp = "2022-12-31",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    split_col: str = "split_fe",
    sort_before_split: bool = True,
    overwrite: bool = True,
    print_report: bool = True,
) -> pd.DataFrame:
    """
    Agrega una columna de split IS/OOS al dataset de entrada.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset de entrada, en este caso mnq_intraday_with_indicators.
    cut_date : str | pd.Timestamp
        Fecha de corte. <= cut_date -> IS, > cut_date -> OOS.
    date_col : str
        Nombre de la columna de fecha.
    minute_col : str
        Nombre de la columna de minuto intradía.
    split_col : str
        Nombre de la columna de salida para el split.
    sort_before_split : bool
        Si True, ordena cronológicamente antes de crear el split.
    overwrite : bool
        Si False y split_col ya existe, lanza error.
    print_report : bool
        Si True, imprime un resumen del split.

    Devuelve
    --------
    pd.DataFrame
        DataFrame con la columna split_fe agregada.
    """
    out = df.copy()

    if split_col in out.columns and not overwrite:
        raise ValueError(f"La columna '{split_col}' ya existe y overwrite=False.")

    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' en el DataFrame.")

    out[date_col] = pd.to_datetime(out[date_col])
    cut_date = pd.to_datetime(cut_date)

    # ============================================================
    # 1) Orden cronológico
    # ============================================================
    if sort_before_split:
        if isinstance(out.index, pd.DatetimeIndex):
            out = out.sort_index().copy()
        else:
            sort_cols = [date_col]
            if minute_col in out.columns:
                sort_cols.append(minute_col)
            out = out.sort_values(by=sort_cols).copy()

    # Validación
    if isinstance(out.index, pd.DatetimeIndex):
        is_sorted = out.index.is_monotonic_increasing
    else:
        if minute_col in out.columns:
            is_sorted = (
                out[[date_col, minute_col]]
                .reset_index(drop=True)
                .equals(
                    out[[date_col, minute_col]]
                    .sort_values([date_col, minute_col])
                    .reset_index(drop=True)
                )
            )
        else:
            is_sorted = out[date_col].is_monotonic_increasing

    if not is_sorted:
        raise ValueError("El dataset no está ordenado cronológicamente antes del split.")

    # ============================================================
    # 2) Crear split IS / OOS
    # ============================================================
    out[split_col] = np.where(
        out[date_col] <= cut_date,
        "IS",
        "OOS",
    )

    # ============================================================
    # 3) Reporte
    # ============================================================
    if print_report:
        split_summary = (
            out.groupby(split_col)
            .agg(
                n_rows=(split_col, "size"),
                start_date=(date_col, "min"),
                end_date=(date_col, "max"),
                n_days=(date_col, "nunique"),
            )
            .reset_index()
        )

        print("=" * 90)
        print("SEPARACIÓN IS vs OOS")
        print("=" * 90)
        print(f"Fecha de corte: {cut_date.date()}")
        print(f"Orden cronológico correcto: {is_sorted}")
        print()
        print(split_summary.to_string(index=False))

    return out


In [23]:
mnq_intraday_t2_ti = add_is_oos_split(
    mnq_intraday_t2_ti,
    cut_date="2022-12-31",
    print_report=True,
)

SEPARACIÓN IS vs OOS
Fecha de corte: 2022-12-31
Orden cronológico correcto: True

split_fe  n_rows start_date   end_date  n_days
      IS  426818 2020-01-02 2022-12-30     709
     OOS  352772 2023-01-03 2025-06-13     586


In [24]:
mnq_intraday_t2_ti

,date,open,high,low,close,volume,minute_of_day,regime_id,rsi_14,rsi_7,...,delta_90,t1_dir_bin_90,close_fwd_120,delta_120,t1_dir_bin_120,t2_dir_thr_30,t2_dir_thr_60,t2_dir_thr_90,t2_dir_thr_120,split_fe
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-02 05:59:00-05:00,2020-01-02,8819.50,8819.50,8818.50,8818.75,42,359,0,45.868327,37.935475,...,-4.50,0.0,8817.50,-1.25,0.0,0.0,0.0,0.0,0.0,IS
2020-01-02 06:00:00-05:00,2020-01-02,8818.50,8818.75,8818.00,8818.00,70,360,0,41.589146,31.183907,...,-3.50,0.0,8818.50,0.50,1.0,0.0,0.0,0.0,0.0,IS
2020-01-02 06:01:00-05:00,2020-01-02,8818.00,8818.25,8817.75,8817.75,20,361,0,40.241471,29.165302,...,-3.50,0.0,8818.75,1.00,1.0,0.0,0.0,0.0,0.0,IS
2020-01-02 06:02:00-05:00,2020-01-02,8817.50,8817.75,8816.50,8816.50,81,362,0,34.263058,21.171024,...,-1.25,0.0,8820.00,3.50,1.0,0.0,0.0,0.0,0.0,IS
2020-01-02 06:03:00-05:00,2020-01-02,8816.75,8817.50,8816.75,8817.50,28,363,0,41.722215,37.229525,...,-2.50,0.0,8818.75,1.25,1.0,0.0,0.0,0.0,0.0,IS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,4,42.096236,39.066557,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OOS
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,4,45.107585,45.637235,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OOS
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,4,44.313396,43.871077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OOS


#**4. Evaluación mediante Information Coefficient (IC)**

## **4.1. Marco teórico**

### **4.1.1. Target T2 (dirección con umbral)**

En el dataset se utilizan como variables objetivo los targets discretos T2, definidos como:

- t2_dir_thr_h ∈ { -1, 0, 1 }

donde:

- 1  → movimiento positivo significativo  
- -1 → movimiento negativo significativo  
- 0  → movimiento no significativo (ruido)

Estos targets se construyen a partir de los deltas futuros (delta_h), aplicando un umbral definido mediante percentiles de la distribución de |delta_h|.

A diferencia de los targets continuos (delta_h), T2 no busca modelar la magnitud exacta del movimiento, sino identificar si el movimiento es lo suficientemente relevante como para ser considerado potencialmente operable.

De esta forma, T2 permite:

- filtrar el ruido de baja magnitud,
- concentrar el análisis en eventos significativos,
- mejorar la relación señal/ruido del problema.

---

Para evaluar la relación entre los indicadores técnicos y el target T2, se utiliza el coeficiente de correlación de Spearman.

Aunque T2 es una variable discreta, el uso de Spearman sigue siendo adecuado porque:

- no asume relaciones lineales,
- es robusto a outliers,
- captura relaciones monotónicas entre variables continuas (indicadores) y el target ordinal (-1 < 0 < 1).

---

Interpretación del IC (Information Coefficient):

- IC > 0  
  → el indicador tiende a tomar valores altos cuando ocurren movimientos positivos significativos (clase 1)

- IC < 0  
  → el indicador tiende a tomar valores altos cuando ocurren movimientos negativos significativos (clase -1)

- IC ≈ 0  
  → el indicador no muestra relación clara con el target

---

Es importante destacar que, en este contexto, el IC no mide la capacidad de predecir con precisión la clase, sino la existencia de una relación estadística entre el indicador y la ocurrencia de movimientos significativos.

Por lo tanto, valores de IC relativamente bajos (por ejemplo, en el rango 0.02–0.05) pueden ser relevantes en el contexto de datos financieros intradía.


### **4.1.2. Criterio metodológico — Evaluación por régimen e IS/OOS**


El IC se calcula de forma segmentada considerando dos dimensiones:

- régimen de mercado (según `regime_id`)
- partición temporal (`split_fe`: IS / OOS)

Este enfoque permite:

- identificar factores que mantienen señal en distintos contextos intradía,
- detectar dependencias específicas por régimen (ej: opening vs overnight),
- evaluar la estabilidad de la señal fuera de muestra (OOS),
- evitar sobreajuste a condiciones particulares del mercado.

De esta forma, cada indicador es evaluado no solo por su capacidad predictiva, sino por su **robustez estructural**.

### **4.1.3. Uso de los resultados**

Las tablas de IC obtenidas permiten:

- identificar indicadores con señal consistente en OOS,
- comparar el comportamiento de los indicadores entre regímenes,
- evaluar estabilidad entre IS y OOS (gap de generalización),
- detectar indicadores inestables o dependientes de condiciones específicas.

En particular, se priorizan indicadores que:

- mantienen signo consistente entre IS y OOS,
- presentan diferencias pequeñas entre ambos (bajo gap),
- muestran señal en múltiples regímenes.

### **4.1.4. Síntesis**


El Information Coefficient permite medir de forma directa la relación entre los indicadores técnicos y la ocurrencia de movimientos significativos del mercado.

Su evaluación conjunta por régimen de mercado y partición temporal (IS/OOS) constituye un criterio fundamental para la selección de factores:

- robustos,
- consistentes,
- y generalizables.

Este proceso permite reducir el conjunto de indicadores, eliminando redundancia y priorizando aquellos con mayor capacidad de aportar señal real al modelo.

## **4.2. Implementación de cálculo de IC**

### **4.2.1. Marcas temporales de régimen**

In [26]:
import pandas as pd


def build_regime_ranges_table(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    regime_col: str = "regime_id",
    regime_mapping: dict | None = None,
) -> pd.DataFrame:
    """
    Construye una tabla resumen por régimen (regime_id) con:

    - régimen (id o nombre)
    - minuto inicial
    - minuto final
    - hora inicial HH:MM
    - hora final HH:MM
    - cantidad de filas
    """

    if minute_col not in df.columns:
        raise KeyError(f"Falta la columna '{minute_col}' en el DataFrame.")

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    rows = []

    # recorrer cada régimen único
    for regime_id in sorted(df[regime_col].dropna().unique()):
        m = df.loc[df[regime_col] == regime_id, minute_col].dropna()

        if m.empty:
            continue

        start_min = int(m.min())
        end_min = int(m.max())

        # nombre del régimen (opcional)
        regime_name = (
            regime_mapping.get(regime_id, regime_id)
            if regime_mapping is not None
            else regime_id
        )

        rows.append({
            "regime_id": regime_id,
            "regime_name": regime_name,
            "start_minute_of_day": start_min,
            "end_minute_of_day": end_min,
            "start_hhmm": f"{start_min // 60:02d}:{start_min % 60:02d}",
            "end_hhmm": f"{end_min // 60:02d}:{end_min % 60:02d}",
            "n_rows": int(m.shape[0]),
        })

    out = pd.DataFrame(rows)

    out = out.sort_values(
        by="start_minute_of_day"
    ).reset_index(drop=True)

    return out[
        [
            "regime_id",
            "regime_name",
            "start_minute_of_day",
            "end_minute_of_day",
            "start_hhmm",
            "end_hhmm",
            "n_rows",
        ]
    ]

In [28]:
regime_mapping = {
    0: "overnight",
    1: "premarket",
    2: "opening",
    3: "regular",
    4: "closing",
}

regime_table = build_regime_ranges_table(
    mnq_intraday_t2_ti,
    regime_mapping=regime_mapping,
)

regime_table

,regime_id,regime_name,start_minute_of_day,end_minute_of_day,start_hhmm,end_hhmm,n_rows
0,0,overnight,359,960,05:59,16:00,196840
1,1,premarket,510,569,08:30,09:29,77700
2,2,opening,570,629,09:30,10:29,77700
3,3,regular,630,929,10:30,15:29,388500
4,4,closing,930,959,15:30,15:59,38850


### **4.2.2. Funciones para calcular IC**

In [32]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# =========================
# Utilidades base
# =========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """IC Spearman entre x e y, ignorando NaNs."""
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_market_regime(
    df: pd.DataFrame,
    regime_id: int | None = None,
    regime_col: str = "regime_id",
) -> pd.DataFrame:
    """
    Filtra el DataFrame por regime_id.

    Parámetros
    ----------
    regime_id : int | None
        - None -> no filtra, usa toda la jornada
        - int  -> usa solo filas con ese regime_id
    regime_col : str
        Nombre de la columna de régimen.
    """
    if regime_id is None:
        return df.copy()

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    return df.loc[df[regime_col] == regime_id].copy()


def daily_ic(
    df: pd.DataFrame,
    indicator_col: str,
    target_col: str,
    date_col: str = "date",
) -> pd.Series:
    """IC Spearman por día."""
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    return df.groupby(date_col).apply(
        lambda g: spearman_ic(g[indicator_col], g[target_col])
    )


# =========================
# Tabla IC (indicadores × T2) con IS/OOS por régimen
# =========================

def compute_ic_table_is_oos_by_regime_t2(
    df: pd.DataFrame,
    indicator_columns: list[str],
    *,
    horizons: tuple[int, ...] = (30, 60, 90, 120),
    regime_id: int | None = None,         # None = full day
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",          # "IS" / "OOS"
    date_col: str = "date",
    target_prefix: str = "t2_dir_thr_",
) -> pd.DataFrame:
    """
    Calcula IC Spearman entre cada indicador y los targets T2,
    separando IS y OOS, para toda la jornada o para un régimen específico.

    Targets esperados:
      - t2_dir_thr_30
      - t2_dir_thr_60
      - t2_dir_thr_90
      - t2_dir_thr_120

    Devuelve:
      - indicator, horizon, target_col, regime_id
      - IC_IS, IC_OOS, oos_minus_is
      - n_pairs_IS, n_pairs_OOS
      - abs_IC_OOS
      - note
    """
    # 1) Filtrado por régimen
    dfr = filter_market_regime(df, regime_id=regime_id, regime_col=regime_col)

    # 2) Validaciones estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfr.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfr[dfr[split_col] == "IS"].copy()
    df_oos = dfr[dfr[split_col] == "OOS"].copy()

    rows = []

    for ind in indicator_columns:
        for h in horizons:
            target_col = f"{target_prefix}{h}"

            # Validación de columnas
            missing = [c for c in (ind, target_col) if c not in dfr.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "target_col": target_col,
                    "regime_id": regime_id if regime_id is not None else "full_day",
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "abs_IC_OOS": np.nan,
                    "note": f"missing: {missing}",
                })
                continue

            # Conteo de pares válidos
            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            # --- IS ---
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = float(ic_is_series.mean()) if len(ic_is_series) > 0 else np.nan
            else:
                ic_is = float(spearman_ic(df_is[ind], df_is[target_col]))

            # --- OOS ---
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = float(ic_oos_series.mean()) if len(ic_oos_series) > 0 else np.nan
            else:
                ic_oos = float(spearman_ic(df_oos[ind], df_oos[target_col]))

            rows.append({
                "indicator": ind,
                "horizon": h,
                "target_col": target_col,
                "regime_id": regime_id if regime_id is not None else "full_day",
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_is) and pd.notna(ic_oos)) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "abs_IC_OOS": abs(ic_oos) if pd.notna(ic_oos) else np.nan,
                "note": "",
            })

    out = pd.DataFrame(rows)

    out = (
        out.sort_values(
            ["regime_id", "horizon", "abs_IC_OOS"],
            ascending=[True, True, False]
        )
        .reset_index(drop=True)
    )

    return out

### **4.2.3. Función para calculo de IC tables**

In [31]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (T2)
# ============================================================

CACHE_DIR_T2 = "/content/drive/MyDrive/neural_profit/data/03_targets/ic_tables_t2"
os.makedirs(CACHE_DIR_T2, exist_ok=True)


def _select_cache_dir() -> str:
    """
    Devuelve el directorio de cache para IC tables de T2.
    """
    return CACHE_DIR_T2


def _ic_cache_paths(cache_dir: str, name: str) -> dict:
    """
    Devuelve paths de cache para una ic_table:
      - parquet: datos
      - json: metadatos
    """
    return {
        "data": os.path.join(cache_dir, f"{name}.parquet"),
        "meta": os.path.join(cache_dir, f"{name}.meta.json"),
    }


def load_or_compute_ic_table_t2(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (30, 60, 90, 120),
    regime_id: int | None = None,   # None = full day
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    target_prefix: str = "t2_dir_thr_",
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga una ic_table T2 desde cache si existe; si no, la calcula y la guarda.

    Parámetros
    ----------
    name : str
        Identificador del artefacto en cache.
    df : pd.DataFrame
        DataFrame con indicadores + targets + columnas auxiliares.
    indicator_columns : list[str]
        Lista de indicadores a evaluar.
    horizons : tuple[int, ...]
        Horizontes a evaluar. Ej.: (30, 60, 90, 120).
    regime_id : int | None
        Régimen de mercado a evaluar.
        - None -> jornada completa
        - int  -> un régimen específico
    regime_col : str
        Nombre de la columna de régimen.
    use_daily_ic : bool
        Si True, calcula IC diario y luego promedia.
        Si False, calcula IC global.
    split_col : str
        Columna de split IS/OOS.
    date_col : str
        Columna de fecha para agrupar daily IC.
    target_prefix : str
        Prefijo de los targets T2.
    force_recompute : bool
        Si True, ignora cache y recalcula.
    """
    cache_dir = _select_cache_dir()
    paths = _ic_cache_paths(cache_dir, name)

    # 1) Cargar desde cache
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table
    ic_table = compute_ic_table_is_oos_by_regime_t2(
        df=df,
        indicator_columns=indicator_columns,
        horizons=horizons,
        regime_id=regime_id,
        regime_col=regime_col,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
        target_prefix=target_prefix,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos
    meta = {
        "name": name,
        "target_type": "t2",
        "target_prefix": target_prefix,
        "horizons": list(horizons),
        "regime_id": regime_id if regime_id is not None else "full_day",
        "regime_col": regime_col,
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
        "cache_dir": cache_dir,
    }

    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table

### **4.2.4. Aplicación de cálculos**

In [33]:
def build_all_ic_tables_t2(
    df: pd.DataFrame,
    *,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (30, 60, 90, 120),
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    target_prefix: str = "t2_dir_thr_",
    force_recompute: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Calcula o carga todas las tablas IC para T2:
    - full_day
    - una tabla por cada regime_id disponible en el dataset
    """

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    regime_ids = sorted(df[regime_col].dropna().unique().tolist())

    regime_map = {"full_day": None}
    regime_map.update({f"regime_{int(rid)}": int(rid) for rid in regime_ids})

    ic_tables = {}

    for name, regime_id in regime_map.items():
        ic_tables[name] = load_or_compute_ic_table_t2(
            name=f"ic_table_t2_{name}",
            df=df,
            indicator_columns=indicator_columns,
            horizons=horizons,
            regime_id=regime_id,
            regime_col=regime_col,
            use_daily_ic=use_daily_ic,
            split_col=split_col,
            date_col=date_col,
            target_prefix=target_prefix,
            force_recompute=force_recompute,
        )

    return ic_tables

In [34]:
ic_tables_t2 = build_all_ic_tables_t2(
    mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    horizons=(30, 60, 90, 120),
    regime_col="regime_id",
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
    target_prefix="t2_dir_thr_",
    force_recompute=False,
)

[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_full_day.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_regime_0.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_regime_1.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_regime_2.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_regime_3.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_t2/ic_table_t2_regime_4.parquet


In [35]:
ic_tables_t2.keys()

dict_keys(['full_day', 'regime_0', 'regime_1', 'regime_2', 'regime_3', 'regime_4'])

In [ ]:
'''
regime_name_map = {
    0: "overnight",
    1: "premarket",
    2: "opening",
    3: "regular",
    4: "closing",
}
'''

In [36]:
ic_table_full_day       = ic_tables_t2["full_day"]
ic_table_overnight    = ic_tables_t2["regime_0"]
ic_table_premarket  = ic_tables_t2["regime_1"]
ic_table_opening    = ic_tables_t2["regime_2"]
ic_table_regular    = ic_tables_t2["regime_3"]
ic_table_closing    = ic_tables_t2["regime_4"]

## **4.3. Análisis de resultados**

Qué vamos a analizar primero:

- Magnitud del IC_OOS → ¿hay señal?
- Signo del IC_OOS → ¿momentum o reversión?
- Top indicadores por régimen → ¿se repiten o cambian?

Sirve para responder rápido cosas como:

- ¿en qué régimen hay más señal OOS?
- ¿predominan relaciones positivas o negativas?
- ¿cuáles son los indicadores más fuertes en cada régimen?

Qué no hace
No te dice todavía:
- si el indicador es robusto IS vs OOS
- si hay redundancia entre indicadores
- si un indicador sirve en varios regímenes

O sea: sirve para exploración inicial, no para selección final.

### **Código**


In [39]:
import pandas as pd


def quick_ic_observation_t2(
    ic_tables: dict[str, pd.DataFrame],
    *,
    horizon: int,
    top_n: int = 5,
) -> None:
    """
    Resumen exploratorio rápido de tablas IC para targets T2.

    Para un horizonte dado:
    - muestra top indicadores por |IC_OOS|
    - resume signo predominante
    - resume magnitud promedio de IC_OOS

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Diccionario de tablas IC, por ejemplo:
        {
            "full_day": ...,
            "regime_0": ...,
            "regime_1": ...,
            ...
        }
    horizon : int
        Horizonte a analizar, por ejemplo 30, 60, 90 o 120.
    top_n : int
        Número de indicadores top a mostrar por tabla.
    """

    print("\n" + "=" * 90)
    print(f"ANÁLISIS SIMPLE IC | T2 horizon={horizon}")
    print("=" * 90)

    for regime_name, df in ic_tables.items():
        x = df[df["horizon"] == horizon].copy()
        x = x[x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        top = x.sort_values("abs_IC_OOS", ascending=False).head(top_n)

        mean_ic = x["IC_OOS"].mean()
        mean_abs_ic = x["abs_IC_OOS"].mean()

        pct_positive = (x["IC_OOS"] > 0).mean() * 100
        pct_negative = (x["IC_OOS"] < 0).mean() * 100
        pct_zero = (x["IC_OOS"] == 0).mean() * 100

        print("\n" + "-" * 90)
        print(f"Régimen: {regime_name}")
        print("-" * 90)

        print(f"IC_OOS medio    : {mean_ic:.4f}")
        print(f"|IC_OOS| medio  : {mean_abs_ic:.4f}")
        print(f"% IC positivo   : {pct_positive:.1f}%")
        print(f"% IC negativo   : {pct_negative:.1f}%")
        print(f"% IC cero       : {pct_zero:.1f}%")

        print("\nTop indicadores por |IC_OOS|:")
        print(
            top[
                [
                    "indicator",
                    "target_col",
                    "IC_OOS",
                    "abs_IC_OOS",
                ]
            ].to_string(index=False)
        )

In [43]:
quick_ic_observation_t2(
    ic_tables_t2,
    horizon=30,
    top_n=10,
)


ANÁLISIS SIMPLE IC | T2 horizon=30

------------------------------------------------------------------------------------------
Régimen: full_day
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0205
|IC_OOS| medio  : 0.0353
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
  indicator    target_col    IC_OOS  abs_IC_OOS
     ema_60 t2_dir_thr_30 -0.063846    0.063846
     roc_60 t2_dir_thr_30 -0.061718    0.061718
atr_norm_30 t2_dir_thr_30  0.061672    0.061672
atr_norm_20 t2_dir_thr_30  0.059002    0.059002
atr_norm_14 t2_dir_thr_30  0.056863    0.056863
atr_norm_10 t2_dir_thr_30  0.055020    0.055020
 atr_norm_5 t2_dir_thr_30  0.051180    0.051180
   bb_60_15 t2_dir_thr_30 -0.050397    0.050397
   bb_60_20 t2_dir_thr_30 -0.050397    0.050397
   bb_60_25 t2_dir_thr_30 -0.050397    0.050397

---------------------------------------------------------------------------------

In [40]:
quick_ic_observation_t2(
    ic_tables_t2,
    horizon=60,
    top_n=10,
)


ANÁLISIS SIMPLE IC | T2 horizon=60

------------------------------------------------------------------------------------------
Régimen: full_day
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0266
|IC_OOS| medio  : 0.0440
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
  indicator    target_col    IC_OOS  abs_IC_OOS
     ema_60 t2_dir_thr_60 -0.087846    0.087846
     roc_60 t2_dir_thr_60 -0.084619    0.084619
atr_norm_30 t2_dir_thr_60  0.070921    0.070921
atr_norm_20 t2_dir_thr_60  0.069299    0.069299
atr_norm_14 t2_dir_thr_60  0.067308    0.067308
   bb_60_15 t2_dir_thr_60 -0.066377    0.066377
   bb_60_20 t2_dir_thr_60 -0.066377    0.066377
   bb_60_25 t2_dir_thr_60 -0.066377    0.066377
atr_norm_10 t2_dir_thr_60  0.065295    0.065295
     roc_30 t2_dir_thr_60 -0.062878    0.062878

---------------------------------------------------------------------------------

In [41]:
quick_ic_observation_t2(
    ic_tables_t2,
    horizon=90,
    top_n=10,
)


ANÁLISIS SIMPLE IC | T2 horizon=90

------------------------------------------------------------------------------------------
Régimen: full_day
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0365
|IC_OOS| medio  : 0.0549
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
  indicator    target_col    IC_OOS  abs_IC_OOS
     ema_60 t2_dir_thr_90 -0.113862    0.113862
     roc_60 t2_dir_thr_90 -0.108494    0.108494
   bb_60_15 t2_dir_thr_90 -0.083649    0.083649
   bb_60_20 t2_dir_thr_90 -0.083649    0.083649
   bb_60_25 t2_dir_thr_90 -0.083649    0.083649
     roc_30 t2_dir_thr_90 -0.081582    0.081582
     ema_30 t2_dir_thr_90 -0.080903    0.080903
     rsi_14 t2_dir_thr_90 -0.076423    0.076423
 stoch_k_30 t2_dir_thr_90 -0.071867    0.071867
atr_norm_20 t2_dir_thr_90  0.071351    0.071351

---------------------------------------------------------------------------------

In [42]:
quick_ic_observation_t2(
    ic_tables_t2,
    horizon=120,
    top_n=10,
)


ANÁLISIS SIMPLE IC | T2 horizon=120

------------------------------------------------------------------------------------------
Régimen: full_day
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0505
|IC_OOS| medio  : 0.0676
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
 indicator     target_col    IC_OOS  abs_IC_OOS
    ema_60 t2_dir_thr_120 -0.148192    0.148192
    roc_60 t2_dir_thr_120 -0.146961    0.146961
    ema_30 t2_dir_thr_120 -0.107633    0.107633
  bb_60_15 t2_dir_thr_120 -0.105692    0.105692
  bb_60_20 t2_dir_thr_120 -0.105692    0.105692
  bb_60_25 t2_dir_thr_120 -0.105692    0.105692
    roc_30 t2_dir_thr_120 -0.104609    0.104609
    rsi_14 t2_dir_thr_120 -0.098142    0.098142
stoch_k_30 t2_dir_thr_120 -0.091453    0.091453
    ema_20 t2_dir_thr_120 -0.090067    0.090067

--------------------------------------------------------------------------------

### **Observaciones**


El análisis del Information Coefficient (IC) aplicado a los targets T2 permite extraer varias conclusiones claras sobre la existencia, naturaleza y localización de la señal en el problema.

- En primer lugar, existe evidencia de señal en los datos, pero esta no es homogénea a lo largo de toda la jornada. Cuando se analiza el dataset completo (full_day), los valores de |IC_OOS| son relativamente bajos (en el rango aproximado de 0.03 a 0.06), lo que podría sugerir una señal débil. Sin embargo, al segmentar por régimen de mercado, se observa que en ciertos regímenes —especialmente regime_2— la magnitud del IC aumenta de forma significativa, alcanzando valores superiores a 0.20 e incluso cercanos a 0.50. Esto indica que la señal no es inexistente, sino altamente dependiente del contexto intradía.

- En segundo lugar, se observa un predominio claro de IC negativos en todos los horizontes y regímenes, con porcentajes que oscilan entre el 75% y el 88%. Esto implica que, en general, valores altos de los indicadores técnicos se asocian con movimientos negativos significativos del mercado, y viceversa. Esta relación es consistente con un comportamiento de reversión a la media en el corto plazo, lo cual es coherente con la dinámica típica de datos intradía.

- En tercer lugar, los indicadores que aparecen en los primeros lugares del ranking son recurrentes a lo largo de todos los horizontes y regímenes. Entre ellos destacan las medias móviles exponenciales (ema_60, ema_30, ema_20), los indicadores de momentum (roc_60, roc_30, roc_20), así como rsi_14, stoch_k_30 y las bandas de Bollinger. Esta repetición sistemática sugiere una alta redundancia entre indicadores, ya que muchos pertenecen a la misma familia y capturan esencialmente la misma información de mercado.

- En cuarto lugar, los indicadores basados en volatilidad, particularmente atr_norm_*, presentan consistentemente IC positivos. Esto sugiere que niveles elevados de volatilidad están asociados con una mayor probabilidad de observar movimientos significativos (clases ±1 en T2), lo cual es coherente con la construcción del target basada en umbrales.

- En quinto lugar, se observa que la magnitud del IC tiende a incrementarse con el horizonte temporal. Para horizontes cortos (30 minutos), la señal es más débil, mientras que para horizontes mayores (90 y 120 minutos) la relación entre indicadores y target se vuelve más fuerte. Esto indica que el mercado requiere cierto tiempo para que las señales capturadas por los indicadores se materialicen en movimientos significativos.

- En sexto lugar, el régimen identificado como regime_2 destaca de manera consistente como el entorno con mayor señal. Los valores de IC en este régimen son significativamente superiores al resto, lo que indica que este segmento del día concentra gran parte del edge del sistema. Esto sugiere que modelar explícitamente por régimen no solo es recomendable, sino necesario para capturar la estructura real del mercado.

Finalmente, el análisis sobre la jornada completa (full_day) tiende a diluir la señal debido a la mezcla de distintos regímenes con dinámicas heterogéneas. Como consecuencia, un modelo entrenado sin segmentación por régimen probablemente subestime la verdadera capacidad predictiva de los indicadores.

En conjunto, estos resultados indican que:

- la señal existe, pero es contextual y no global,
- la dinámica predominante es de reversión a la media,
- existe redundancia significativa entre indicadores técnicos,
- la volatilidad juega un rol relevante en la ocurrencia de eventos,
- la señal mejora con horizontes más largos,
- y la segmentación por régimen es un componente crítico para capturar el edge del problema.

Estas conclusiones justifican avanzar hacia una etapa de selección de features que priorice indicadores robustos, no redundantes y consistentes entre regímenes y fuera de muestra.

# **5. Análisis de robustez IS vs OOS**


El objetivo de este análisis es evaluar la estabilidad de la señal de los indicadores técnicos al pasar de datos in-sample (IS) a out-of-sample (OOS).

En el contexto de machine learning aplicado a trading, este punto es crítico, ya que un indicador puede mostrar una relación aparente fuerte en entrenamiento, pero perder completamente su capacidad predictiva al generalizar. Este comportamiento es característico del sobreajuste (overfitting), y es precisamente lo que se busca detectar en esta etapa.

---

Para ello, se compara el Information Coefficient (IC) calculado en IS y en OOS para cada indicador, horizonte y régimen de mercado.

La lógica del análisis se basa en los siguientes criterios:

- Diferencia entre IC_IS e IC_OOS (gap)
  
  Se define como:
  
  gap = IC_OOS - IC_IS

  Interpretación:
  
  - gap pequeño → la señal se mantiene al pasar a OOS (mayor robustez)
  - gap grande → la señal cambia significativamente (posible inestabilidad)

---

- Magnitud del gap (abs_gap)

  abs_gap = |IC_OOS - IC_IS|

  Este valor permite medir directamente la estabilidad de la señal sin considerar el signo.

  Interpretación:
  
  - abs_gap bajo → indicador estable
  - abs_gap alto → indicador inestable

---

- Consistencia de signo entre IS y OOS

  Se evalúa si el signo del IC se mantiene:

  - mismo signo → la relación entre indicador y target se conserva
  - cambio de signo → la señal es inconsistente o potencialmente espuria

  Este criterio es especialmente importante en targets T2, ya que el signo del IC define si el indicador está asociado a movimientos positivos o negativos significativos.

---

- IC_OOS como referencia principal

  Aunque se analiza la relación entre IS y OOS, el valor más importante es IC_OOS, ya que representa la capacidad predictiva fuera de muestra.

  Por lo tanto:

  - IC_OOS alto + bajo abs_gap → indicador robusto
  - IC_OOS alto + alto abs_gap → indicador sospechoso
  - IC_OOS bajo → indicador poco útil, independientemente de su estabilidad

---

Este análisis permite:

- identificar indicadores que mantienen su señal fuera de muestra,
- descartar indicadores que dependen del dataset de entrenamiento,
- priorizar factores robustos y generalizables,
- reducir el riesgo de sobreajuste en etapas posteriores de modelado.

---

En conjunto, la robustez IS vs OOS constituye un criterio fundamental para la selección de features en problemas financieros, donde la capacidad de generalización es más importante que el ajuste en entrenamiento.

## **5.1. Implementación de código**

In [45]:
import numpy as np
import pandas as pd

def analyze_ic_robustness(
    ic_table: pd.DataFrame,
    *,
    regime_name: str,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
    top_n: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza robustez IS vs OOS para una tabla IC de un régimen.

    Devuelve:
    ---------
    {
        "summary_df": resumen por horizonte,
        "top_df": top indicadores robustos por horizonte
    }
    """

    required = [
        "indicator", "horizon", "target_col",
        "IC_IS", "IC_OOS"
    ]
    missing = [c for c in required if c not in ic_table.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    df = ic_table.copy()
    df = df[df["IC_IS"].notna() & df["IC_OOS"].notna()].copy()

    if df.empty:
        empty_summary = pd.DataFrame([{
            "regime_name": regime_name,
            "horizon": pd.NA,
            "n_total": 0,
            "n_after_filter": 0,
            "pct_retained": np.nan,
            "mean_IC_IS": np.nan,
            "mean_IC_OOS": np.nan,
            "mean_abs_IC_OOS": np.nan,
            "mean_abs_gap": np.nan,
            "pct_same_sign": np.nan,
            "best_indicator": pd.NA,
            "best_IC_OOS": np.nan,
            "best_abs_gap": np.nan,
            "best_robustness_score": np.nan,
        }])
        empty_top = pd.DataFrame(columns=[
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ])
        return {"summary_df": empty_summary, "top_df": empty_top}

    # Variables base
    df["oos_minus_is"] = df["IC_OOS"] - df["IC_IS"]
    df["abs_gap"] = df["oos_minus_is"].abs()
    df["same_sign"] = np.sign(df["IC_IS"]) == np.sign(df["IC_OOS"])
    df["abs_IC_IS"] = df["IC_IS"].abs()
    df["abs_IC_OOS"] = df["IC_OOS"].abs()

    df["strength_ratio"] = np.where(
        df["abs_IC_IS"] > 1e-12,
        df["abs_IC_OOS"] / df["abs_IC_IS"],
        np.nan,
    )

    sign_bonus = np.where(df["same_sign"], 1.0, 0.0)

    # Score robustez
    df["robustness_score"] = (
        0.70 * df["abs_IC_OOS"]
        - 0.25 * df["abs_gap"]
        + 0.05 * sign_bonus
    )

    # Conteo previo por horizonte
    n_total_by_h = (
        df.groupby("horizon")
          .size()
          .rename("n_total")
          .reset_index()
    )

    # Filtro mínimo de señal
    df = df[df["abs_IC_OOS"] >= min_abs_ic_oos].copy()

    # Filtro opcional por consistencia de signo
    if require_same_sign:
        df = df[df["same_sign"]].copy()

    # Si tras filtrar queda vacío
    if df.empty:
        summary_rows = []
        for _, row in n_total_by_h.iterrows():
            summary_rows.append({
                "regime_name": regime_name,
                "horizon": row["horizon"],
                "n_total": int(row["n_total"]),
                "n_after_filter": 0,
                "pct_retained": 0.0,
                "mean_IC_IS": np.nan,
                "mean_IC_OOS": np.nan,
                "mean_abs_IC_OOS": np.nan,
                "mean_abs_gap": np.nan,
                "pct_same_sign": np.nan,
                "best_indicator": pd.NA,
                "best_IC_OOS": np.nan,
                "best_abs_gap": np.nan,
                "best_robustness_score": np.nan,
            })
        return {
            "summary_df": pd.DataFrame(summary_rows),
            "top_df": pd.DataFrame(columns=[
                "regime_name", "indicator", "horizon", "target_col",
                "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
                "same_sign", "abs_IC_OOS", "strength_ratio",
                "robustness_score"
            ])
        }

    # Orden final
    df = df.sort_values(
        ["horizon", "robustness_score", "abs_IC_OOS", "abs_gap"],
        ascending=[True, False, False, True]
    ).reset_index(drop=True)

    df["regime_name"] = regime_name

    # Top robustos por horizonte
    top_df = (
        df.groupby("horizon", group_keys=False)
          .head(top_n)
          .reset_index(drop=True)
    )[
        [
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ]
    ]

    # Resumen exacto para conclusiones
    summary_rows = []
    for h, g in df.groupby("horizon"):
        n_total = int(n_total_by_h.loc[n_total_by_h["horizon"] == h, "n_total"].iloc[0])
        best = g.iloc[0]

        summary_rows.append({
            "regime_name": regime_name,
            "horizon": h,
            "n_total": n_total,
            "n_after_filter": int(len(g)),
            "pct_retained": float(len(g) / n_total * 100) if n_total > 0 else np.nan,
            "mean_IC_IS": float(g["IC_IS"].mean()),
            "mean_IC_OOS": float(g["IC_OOS"].mean()),
            "mean_abs_IC_OOS": float(g["abs_IC_OOS"].mean()),
            "mean_abs_gap": float(g["abs_gap"].mean()),
            "pct_same_sign": float(g["same_sign"].mean() * 100),
            "best_indicator": best["indicator"],
            "best_IC_OOS": float(best["IC_OOS"]),
            "best_abs_gap": float(best["abs_gap"]),
            "best_robustness_score": float(best["robustness_score"]),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("horizon").reset_index(drop=True)

    return {
        "summary_df": summary_df,
        "top_df": top_df,
    }

In [46]:
rob_premarket = analyze_ic_robustness(ic_table_premarket, regime_name="premarket")
rob_opening   = analyze_ic_robustness(ic_table_opening,   regime_name="opening")
rob_overnight = analyze_ic_robustness(ic_table_overnight, regime_name="overnight")
rob_regular   = analyze_ic_robustness(ic_table_regular,   regime_name="regular")
rob_closing   = analyze_ic_robustness(ic_table_closing,   regime_name="closing")

## **5.2. Summary all**

In [47]:
robustness_summary_all = pd.concat([
    rob_overnight["summary_df"],
    rob_premarket["summary_df"],
    rob_opening["summary_df"],
    rob_regular["summary_df"],
    rob_closing["summary_df"],
], ignore_index=True)

robustness_summary_all


,regime_name,horizon,n_total,n_after_filter,pct_retained,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator,best_IC_OOS,best_abs_gap,best_robustness_score
0,overnight,30,42,35,83.333333,-0.066081,-0.061596,0.077162,0.006080,100.0,roc_60,-0.146345,0.027687,0.145520
1,overnight,60,42,36,85.714286,-0.110284,-0.074881,0.097057,0.035403,100.0,roc_60,-0.181122,0.062230,0.161228
2,overnight,90,42,35,83.333333,-0.098382,-0.070049,0.091203,0.028333,100.0,ema_60,-0.147680,0.047093,0.141603
3,overnight,120,42,35,83.333333,-0.079959,-0.064977,0.080355,0.015302,100.0,ema_60,-0.126342,0.040657,0.128275
4,premarket,30,42,34,80.952381,-0.105432,-0.127476,0.133802,0.028726,100.0,roc_60,-0.243078,0.068027,0.203148
5,premarket,60,42,32,76.190476,-0.128984,-0.146724,0.146724,0.019215,100.0,roc_60,-0.234321,0.060775,0.198831
6,premarket,90,42,32,76.190476,-0.138756,-0.127208,0.127208,0.013186,100.0,roc_60,-0.222573,0.025328,0.199469
7,premarket,120,42,32,76.190476,-0.170848,-0.130120,0.130120,0.040727,100.0,roc_60,-0.261008,0.004120,0.231675
8,opening,30,42,37,88.095238,-0.221367,-0.234947,0.265248,0.015553,100.0,roc_60,-0.533443,0.003034,0.422652
9,opening,60,42,37,88.095238,-0.242697,-0.237834,0.270364,0.009671,100.0,roc_60,-0.525943,0.003062,0.417395


### **Conclusiones — Análisis de robustez IS vs OOS para targets T2**


- El análisis de robustez IS vs OOS muestra que la señal capturada por los indicadores técnicos es altamente estable al pasar de datos in-sample a out-of-sample. En todos los regímenes y horizontes evaluados, el porcentaje de coincidencia de signo entre IC_IS e IC_OOS es del 100%, lo que indica que la relación entre los indicadores y el target T2 se mantiene consistentemente fuera de muestra. Además, los valores de abs_gap son muy bajos (en el rango aproximado de 0.005 a 0.04), lo que confirma que no existe degradación significativa de la señal. En conjunto, esto sugiere que no hay evidencia relevante de sobreajuste en los indicadores analizados.

- El régimen de mercado más relevante es claramente el correspondiente a "opening". En este segmento, la magnitud de la señal es significativamente superior al resto, con valores de mean_abs_IC_OOS cercanos a 0.26–0.28 y valores máximos de IC_OOS alrededor de -0.53. Esto indica la presencia de un edge fuerte y consistente, lo que convierte a este régimen en el principal candidato para el modelado. La señal en este contexto no solo es intensa, sino también extremadamente estable.

- En segundo lugar, los regímenes de "premarket" y "regular" presentan una señal intermedia. En ambos casos, los valores de |IC_OOS| se sitúan aproximadamente entre 0.08 y 0.15, lo que indica que existe capacidad predictiva, aunque menor que en el régimen de apertura. Estos regímenes siguen siendo relevantes, pero su contribución es secundaria en comparación con "opening".

- El régimen "overnight" presenta la señal más débil, con valores de |IC_OOS| en torno a 0.07–0.10. Sin embargo, esta señal es igualmente estable, con bajo gap y consistencia total de signo. Esto sugiere que, aunque la capacidad predictiva es limitada, el comportamiento es confiable y no aleatorio.

- Al analizar los distintos horizontes (30, 60, 90 y 120 minutos), no se observan diferencias estructurales significativas en la señal. La magnitud del IC y el ranking de indicadores se mantienen relativamente estables entre horizontes. Esto indica que la señal es estructural y no depende de manera crítica del horizonte temporal. En consecuencia, el horizonte no parece ser el factor determinante en la calidad de la señal.

- En contraste, el régimen de mercado sí tiene un impacto decisivo. La variabilidad de la señal entre regímenes es mucho mayor que entre horizontes, lo que sugiere que el contexto intradía es el principal determinante del comportamiento predictivo de los indicadores. Esto refuerza la necesidad de modelar de forma segmentada por régimen.

- En cuanto a los indicadores, se observa una dominancia clara de ciertas familias. En particular, los indicadores de momentum como roc_60, roc_30 y roc_20, junto con las medias móviles exponenciales como ema_60, ema_30 y ema_20, concentran gran parte de la señal. En el régimen de apertura y premarket domina principalmente roc_60, mientras que en los regímenes de regular y overnight predomina ema_60. Esta consistencia indica que la señal está capturada principalmente por estas dos familias, lo que sugiere una alta redundancia entre indicadores.

- La tasa de retención de indicadores tras aplicar los filtros de robustez es elevada, situándose entre el 75% y el 88%. Esto implica que muchos indicadores muestran señal y estabilidad, pero también refuerza la idea de que existe redundancia, ya que múltiples indicadores están capturando la misma estructura subyacente del mercado.

- Otro aspecto relevante es que todos los IC presentan signo negativo de forma consistente. Esto indica que el comportamiento dominante del mercado en este contexto es de reversión a la media: valores altos de los indicadores se asocian sistemáticamente con movimientos negativos futuros significativos, y viceversa. Esta dinámica es coherente con la naturaleza intradía del mercado.

- Finalmente, el régimen de "closing" no presenta observaciones válidas en este análisis, por lo que no puede ser evaluado y puede descartarse en esta etapa.

- En conjunto, los resultados muestran que la señal es real, robusta y consistente fuera de muestra, que está fuertemente condicionada por el régimen de mercado, que presenta una dinámica de reversión a la media, y que está explicada principalmente por un conjunto reducido de indicadores altamente redundantes. Estas conclusiones justifican avanzar hacia una etapa de selección de features orientada a reducir redundancia y conservar únicamente los factores más representativos y robustos.

## **5.3. Top indicadores robustos**

In [48]:
robustness_top_all = pd.concat([
    rob_overnight["top_df"],
    rob_premarket["top_df"],
    rob_opening["top_df"],
    rob_regular["top_df"],
    rob_closing["top_df"],
], ignore_index=True)

#robustness_top_all

def find_common_indicators(robustness_top_all: pd.DataFrame) -> pd.DataFrame:
    df = robustness_top_all.copy()

    # Conteo de aparición por indicador
    summary = (
        df.groupby("indicator")
        .agg(
            n_regimes=("regime_name", "nunique"),
            regimes=("regime_name", lambda x: sorted(set(x))),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_score=("robustness_score", "mean"),
        )
        .reset_index()
        .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
    )

    return summary

common_indicators = find_common_indicators(robustness_top_all)
common_indicators

,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_score
15,roc_60,4,"[opening, overnight, premarket, regular]",-0.273160,0.273160,0.235615
12,ema_60,4,"[opening, overnight, premarket, regular]",-0.260113,0.260113,0.225781
18,stoch_k_30,4,"[opening, overnight, premarket, regular]",-0.231213,0.231213,0.207967
14,roc_30,4,"[opening, overnight, premarket, regular]",-0.219772,0.219772,0.197026
11,ema_30,4,"[opening, overnight, premarket, regular]",-0.209328,0.209328,0.190882
7,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.184426
8,bb_60_20,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.184426
16,rsi_14,4,"[opening, overnight, premarket, regular]",-0.196704,0.196704,0.182700
10,ema_20,3,"[opening, overnight, premarket]",-0.220789,0.220789,0.199798
9,bb_60_25,3,"[overnight, premarket, regular]",-0.146625,0.146625,0.146802


### **Conclusiones — Indicadores técnicos más robustos (análisis cross-regime)**



El análisis conjunto de los indicadores más robustos por régimen permite identificar qué variables mantienen señal de forma consistente a lo largo de distintos contextos de mercado. Este enfoque es especialmente relevante, ya que prioriza indicadores que no dependen de un régimen específico, sino que presentan capacidad predictiva generalizable.

En primer lugar, se observa que un grupo reducido de indicadores aparece de forma consistente en los cuatro regímenes analizados (opening, overnight, premarket y regular). Entre ellos destacan principalmente:

- roc_60  
- ema_60  
- stoch_k_30  
- roc_30  
- ema_30  
- rsi_14  
- bb_60_15 y bb_60_20  

Estos indicadores no solo aparecen en todos los regímenes, sino que además presentan valores elevados de |IC_OOS| y robustness_score, lo que indica que combinan señal fuerte y estabilidad fuera de muestra. En particular, roc_60 y ema_60 lideran el ranking, lo que confirma que el comportamiento del mercado está fuertemente explicado por dinámicas de momentum y tendencia.

En segundo lugar, se observa que muchos de estos indicadores pertenecen a las mismas familias. Por ejemplo:

- roc_* → momentum  
- ema_* → tendencia  
- bb_* → desviación respecto a la media  
- stoch_k → osciladores  
- rsi → sobrecompra/sobreventa  

Esto confirma que existe una redundancia importante entre indicadores, ya que múltiples variables están capturando esencialmente la misma información. Por ejemplo, bb_60_15, bb_60_20 y bb_60_25 presentan valores prácticamente idénticos, lo que sugiere que basta con conservar uno de ellos.

En tercer lugar, algunos indicadores aparecen en tres regímenes, como ema_20 o bb_60_25. Estos también son candidatos válidos, aunque con menor robustez global que los anteriores. A medida que disminuye el número de regímenes en los que aparece un indicador, disminuye su nivel de generalización.

En cuarto lugar, indicadores como roc_20 o algunos bb_* de menor ventana aparecen solo en dos o un régimen. Estos casos deben tratarse con mayor cautela, ya que su señal podría estar asociada a condiciones específicas del mercado y no ser generalizable.

Por otro lado, los indicadores de volatilidad (atr_norm_*) presentan un comportamiento distinto. Estos aparecen únicamente en el régimen "regular" y con signo positivo, lo que indica que capturan información complementaria relacionada con la probabilidad de movimientos significativos. Sin embargo, su baja presencia en otros regímenes sugiere que no son factores universales, sino más bien contextuales.

Un aspecto clave es que todos los indicadores dominantes presentan IC negativos de forma consistente, lo que refuerza la conclusión previa de que la dinámica principal del mercado es de reversión a la media. Esto implica que valores altos de estos indicadores están asociados con movimientos negativos futuros significativos, y viceversa.

En conjunto, este análisis permite concluir que:

- existe un núcleo reducido de indicadores altamente robustos y consistentes entre regímenes,
- la señal está principalmente explicada por factores de momentum y tendencia,
- existe una redundancia significativa entre indicadores dentro de la misma familia,
- algunos indicadores aportan señal adicional en contextos específicos (como la volatilidad en régimen regular),
- y la dinámica dominante del mercado es de reversión a la media.

Estos resultados justifican avanzar hacia una etapa de selección de features que reduzca la redundancia y conserve únicamente un subconjunto representativo de indicadores, priorizando aquellos que:

- aparecen en múltiples regímenes,
- presentan alta magnitud de IC_OOS,
- y mantienen estabilidad entre IS y OOS.

# **6. Consistencia de signo**

## **6.1. Función General**

In [49]:
import numpy as np
import pandas as pd

def analyze_sign_consistency_across_regimes(
    ic_tables: dict[str, pd.DataFrame],
    *,
    top_n_inconsistent: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza la consistencia de signo entre IC_IS e IC_OOS
    para múltiples tablas IC por régimen.

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Diccionario tipo:
        {
            "overnight": ic_table_overnight,
            "premarket": ic_table_premarket,
            ...
        }

    Devuelve
    --------
    {
        "summary_df": resumen por régimen y horizonte,
        "inconsistent_df": indicadores que cambian de signo
    }
    """

    all_rows = []

    for regime_name, df in ic_tables.items():
        required = ["indicator", "horizon", "target_col", "IC_IS", "IC_OOS"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()
        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        all_rows.append(x)

    if not all_rows:
        return {
            "summary_df": pd.DataFrame(),
            "inconsistent_df": pd.DataFrame(),
        }

    all_df = pd.concat(all_rows, ignore_index=True)

    # ============================================================
    # 1) Resumen exacto para conclusiones
    # ============================================================
    summary_df = (
        all_df.groupby(["regime_name", "horizon"], as_index=False)
        .agg(
            n_total=("indicator", "count"),
            n_same_sign=("same_sign", "sum"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
        )
        .sort_values(["horizon", "regime_name"])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) Indicadores inconsistentes
    # ============================================================
    inconsistent_df = (
        all_df[~all_df["same_sign"]]
        .sort_values(["horizon", "abs_IC_OOS", "abs_gap"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    if not inconsistent_df.empty:
        inconsistent_df = (
            inconsistent_df.groupby(["regime_name", "horizon"], group_keys=False)
            .head(top_n_inconsistent)
            .reset_index(drop=True)
        )[
            [
                "regime_name",
                "indicator",
                "horizon",
                "target_col",
                "IC_IS",
                "IC_OOS",
                "oos_minus_is",
                "abs_gap",
                "abs_IC_OOS",
                "same_sign",
            ]
        ]

    return {
        "summary_df": summary_df,
        "inconsistent_df": inconsistent_df,
    }

## **6.2. Aplicarlo a todos los regímenes**

In [50]:
ic_tables = {
    "overnight": ic_table_overnight,
    "premarket": ic_table_premarket,
    "opening": ic_table_opening,
    "regular": ic_table_regular,
    "closing": ic_table_closing,
}

sign_results = analyze_sign_consistency_across_regimes(ic_tables)

sign_summary_df = sign_results["summary_df"]
sign_inconsistent_df = sign_results["inconsistent_df"]

sign_summary_df

,regime_name,horizon,n_total,n_same_sign,pct_same_sign,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,opening,30,42,42,100.000000,-0.188654,-0.204785,0.235864,0.017868
1,overnight,30,42,40,95.238095,-0.056709,-0.051342,0.067585,0.006747
2,premarket,30,42,40,95.238095,-0.084906,-0.099740,0.111772,0.026301
3,regular,30,42,42,100.000000,-0.041778,-0.045876,0.069492,0.004453
4,opening,60,42,42,100.000000,-0.207538,-0.206313,0.241385,0.011579
5,overnight,60,42,40,95.238095,-0.095006,-0.063969,0.084823,0.031037
6,premarket,60,42,40,95.238095,-0.096551,-0.110615,0.114509,0.016169
7,regular,60,42,42,100.000000,-0.059788,-0.056986,0.087511,0.004878
8,opening,90,42,42,100.000000,-0.205253,-0.212622,0.247542,0.014549
9,overnight,90,42,40,95.238095,-0.084235,-0.058690,0.079478,0.025545


## **6.3. Conclusiones — Consistencia de signo IS vs OOS (targets T2)**


El análisis de consistencia de signo entre IC_IS e IC_OOS permite evaluar si la relación entre los indicadores técnicos y el target se mantiene al pasar de entrenamiento a validación fuera de muestra. Este criterio es especialmente relevante, ya que un cambio de signo indica una señal inestable o potencialmente espuria.

En términos generales, los resultados muestran un nivel muy alto de consistencia de signo en todos los regímenes y horizontes. En la mayoría de los casos, el porcentaje de coincidencia de signo (pct_same_sign) se sitúa entre 95% y 100%, lo que indica que la gran mayoría de los indicadores mantienen la misma relación con el target fuera de muestra. Esto refuerza la conclusión de que la señal es estructural y no depende del dataset de entrenamiento.

El régimen de "opening" presenta consistencia perfecta (100%) en todos los horizontes evaluados (30, 60, 90 y 120 minutos). Además, combina esta estabilidad con los mayores valores de IC, lo que confirma que no solo es el régimen con mayor señal, sino también el más confiable desde el punto de vista de generalización.

El régimen "regular" también muestra consistencia perfecta (100%) en todos los horizontes, con valores de gap muy bajos. Esto indica una señal completamente estable, aunque de menor magnitud en comparación con el régimen de apertura.

En los regímenes de "overnight" y "premarket", la consistencia sigue siendo alta, pero no perfecta. En la mayoría de los casos, el porcentaje de coincidencia de signo se sitúa en torno al 95%, lo que implica que uno o dos indicadores por horizonte presentan cambios de signo. Aunque estos casos son minoritarios, deben ser considerados con cautela, ya que pueden introducir ruido en el modelado.

El caso más débil se observa en "premarket" para el horizonte de 120 minutos, donde la consistencia desciende a aproximadamente 88%. Esto indica una mayor inestabilidad en ese contexto específico, lo que sugiere que algunos indicadores pierden coherencia al extender el horizonte temporal en ese régimen.

En cuanto a la magnitud de la señal, los valores de mean_abs_IC_OOS siguen el mismo patrón observado previamente: el régimen de apertura presenta los valores más altos (≈ 0.24–0.25), seguido por premarket (≈ 0.10–0.11), regular (≈ 0.07–0.13) y overnight (≈ 0.06–0.08). Esto confirma que la consistencia de signo no solo es alta, sino que además está asociada a señales económicamente relevantes.

Los valores de mean_abs_gap son bajos en todos los casos, lo que indica que la diferencia entre IC_IS e IC_OOS es pequeña. Esto refuerza la idea de que no hay degradación significativa de la señal fuera de muestra.

En conjunto, este análisis permite concluir que:

- la gran mayoría de los indicadores mantienen su relación con el target fuera de muestra,
- la señal es altamente estable y consistente entre IS y OOS,
- el régimen de apertura es el más robusto tanto en magnitud como en consistencia,
- existen pocos casos de inconsistencia, concentrados principalmente en premarket y overnight,
- y los cambios de signo son minoritarios, pero deben ser considerados en el proceso de selección de features.

Estos resultados confirman que los indicadores técnicos analizados presentan un comportamiento confiable y generalizable, lo que permite avanzar con confianza hacia la etapa de selección final de variables y modelado.

# **7. Análisis por régimen de mercado**


   
   Se evalúan los indicadores en:

* overnight
* premarket
* opening
* regular
* closing

Objetivo:

* detectar factores estructurales
* detectar factores contextuales

## **7.1. Código general**

In [52]:
import numpy as np
import pandas as pd

def analyze_regime_dependence(
    ic_tables: dict[str, pd.DataFrame],
    *,
    horizons: tuple[int, ...] = (30,60, 90,120),
    top_n_each_regime: int = 10,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Analiza factores estructurales y contextuales por régimen de mercado.

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Ejemplo:
        {
            "overnight": ic_table_overnight,
            "premarket": ic_table_premarket,
            "opening": ic_table_opening,
            "regular": ic_table_regular,
            "closing": ic_table_closing,
        }

    horizons : tuple[int, ...]
        Horizontes a evaluar.
    top_n_each_regime : int
        Cantidad de indicadores top por régimen para comparar entre regímenes.
    min_abs_ic_oos : float
        Filtro mínimo de señal OOS.
    require_same_sign : bool
        Si True, exige consistencia de signo IS/OOS.

    Devuelve
    --------
    {
        "summary_by_regime_df": resumen por régimen y horizonte,
        "structural_factors_df": factores estructurales,
        "contextual_factors_df": factores contextuales,
        "top_by_regime_df": top indicadores por régimen
    }
    """

    required_cols = [
        "indicator", "horizon", "target_col",
        "IC_IS", "IC_OOS", "abs_IC_OOS"
    ]

    prepared_rows = []
    top_rows = []

    for regime_name, df in ic_tables.items():
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()

        # filtro mínimo
        x = x[x["abs_IC_OOS"] >= min_abs_ic_oos].copy()

        if require_same_sign:
            x = x[x["same_sign"]].copy()

        if x.empty:
            continue

        prepared_rows.append(x)

        # top por régimen y horizonte
        for h in horizons:
            xh = x[x["horizon"] == h].copy()
            if xh.empty:
                continue

            top_h = (
                xh.sort_values(["abs_IC_OOS", "abs_gap"], ascending=[False, True])
                  .head(top_n_each_regime)
                  .copy()
            )
            top_rows.append(top_h)

    if not prepared_rows:
        return {
            "summary_by_regime_df": pd.DataFrame(),
            "structural_factors_df": pd.DataFrame(),
            "contextual_factors_df": pd.DataFrame(),
            "top_by_regime_df": pd.DataFrame(),
        }

    prepared_df = pd.concat(prepared_rows, ignore_index=True)

    if top_rows:
        top_by_regime_df = pd.concat(top_rows, ignore_index=True)
    else:
        top_by_regime_df = pd.DataFrame()

    # ============================================================
    # 1) Resumen por régimen
    # ============================================================
    summary_by_regime_df = (
        prepared_df.groupby(["regime_name", "horizon"], as_index=False)
        .agg(
            n_indicators=("indicator", "count"),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            best_indicator=("indicator", "first"),
        )
        .sort_values(["horizon", "mean_abs_IC_OOS"], ascending=[True, False])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) Factores estructurales
    #    = aparecen en varios regímenes dentro del top
    # ============================================================
    if not top_by_regime_df.empty:
        structural_factors_df = (
            top_by_regime_df.groupby(["horizon", "indicator"], as_index=False)
            .agg(
                n_regimes=("regime_name", "nunique"),
                regimes=("regime_name", lambda s: sorted(set(s))),
                mean_IC_OOS=("IC_OOS", "mean"),
                mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
                mean_abs_gap=("abs_gap", "mean"),
            )
            .sort_values(["horizon", "n_regimes", "mean_abs_IC_OOS"], ascending=[True, False, False])
            .reset_index(drop=True)
        )
    else:
        structural_factors_df = pd.DataFrame()

    # ============================================================
    # 3) Factores contextuales
    #    = aparecen solo en un régimen dentro del top
    # ============================================================
    if not structural_factors_df.empty:
        contextual_only = structural_factors_df[structural_factors_df["n_regimes"] == 1].copy()

        contextual_factors_df = contextual_only.rename(columns={"regimes": "regime"})[
            ["horizon", "indicator", "regime", "mean_IC_OOS", "mean_abs_IC_OOS", "mean_abs_gap"]
        ].sort_values(["horizon", "mean_abs_IC_OOS"], ascending=[True, False]).reset_index(drop=True)
    else:
        contextual_factors_df = pd.DataFrame()

    return {
        "summary_by_regime_df": summary_by_regime_df,
        "structural_factors_df": structural_factors_df,
        "contextual_factors_df": contextual_factors_df,
        "top_by_regime_df": top_by_regime_df,
    }

In [54]:
ic_tables = {
    "overnight": ic_table_overnight,
    "premarket": ic_table_premarket,
    "opening": ic_table_opening,
    "regular": ic_table_regular,
    "closing": ic_table_closing,
}

regime_analysis = analyze_regime_dependence(
    ic_tables,
    horizons=(30,60, 90,120),
    top_n_each_regime=10,
    min_abs_ic_oos=0.05,
    require_same_sign=True,
)

## **7.2. Resultados**

In [55]:
print('\n1. Resumen por régimen:\n')
display(regime_analysis["summary_by_regime_df"])
print('\n2. Factores estructurales:\n')
display(regime_analysis["structural_factors_df"])
print('\n3. Factores contextuales:\n')
display(regime_analysis["contextual_factors_df"])
print('\n4. Top usados para el análisis:\n')
display(regime_analysis["top_by_regime_df"])


1. Resumen por régimen:



,regime_name,horizon,n_indicators,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator
0,opening,30,37,-0.221367,-0.234947,0.265248,0.015553,100.0,roc_60
1,premarket,30,34,-0.105432,-0.127476,0.133802,0.028726,100.0,roc_60
2,regular,30,32,-0.050233,-0.054862,0.084211,0.005081,100.0,ema_60
3,overnight,30,35,-0.066081,-0.061596,0.077162,0.006080,100.0,roc_60
4,opening,60,37,-0.242697,-0.237834,0.270364,0.009671,100.0,roc_60
5,premarket,60,32,-0.128984,-0.146724,0.146724,0.019215,100.0,roc_60
6,regular,60,35,-0.071907,-0.068046,0.101871,0.004725,100.0,ema_60
7,overnight,60,36,-0.110284,-0.074881,0.097057,0.035403,100.0,roc_60
8,opening,90,37,-0.240209,-0.245503,0.276845,0.013445,100.0,roc_60
9,premarket,90,32,-0.138756,-0.127208,0.127208,0.013186,100.0,roc_60



2. Factores estructurales:



,horizon,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,30,roc_60,4,"[opening, overnight, premarket, regular]",-0.260246,0.260246,0.025428
1,30,ema_60,4,"[opening, overnight, premarket, regular]",-0.244218,0.244218,0.024028
2,30,roc_30,4,"[opening, overnight, premarket, regular]",-0.196340,0.196340,0.018485
3,30,ema_30,4,"[opening, overnight, premarket, regular]",-0.195360,0.195360,0.019324
4,30,rsi_14,4,"[opening, overnight, premarket, regular]",-0.182181,0.182181,0.017873
5,30,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.177207,0.177207,0.022777
6,30,bb_60_20,4,"[opening, overnight, premarket, regular]",-0.177207,0.177207,0.022777
7,30,ema_20,3,"[opening, overnight, premarket]",-0.194452,0.194452,0.018922
8,30,bb_60_25,3,"[overnight, premarket, regular]",-0.125966,0.125966,0.021570
9,30,stoch_k_30,2,"[opening, premarket]",-0.260960,0.260960,0.022234



3. Factores contextuales:



,horizon,indicator,regime,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,30,atr_norm_30,[regular],0.097938,0.097938,0.003460
1,30,atr_norm_20,[regular],0.096469,0.096469,0.002378
2,60,bb_30_15,[premarket],-0.168019,0.168019,0.029948
3,60,atr_norm_14,[regular],0.120809,0.120809,0.002409
4,60,atr_norm_10,[regular],0.120736,0.120736,0.003377
5,120,roc_20,[opening],-0.366180,0.366180,0.008028
6,120,stoch_k_20,[premarket],-0.141257,0.141257,0.032098
7,120,bb_30_20,[premarket],-0.137844,0.137844,0.040861



4. Top usados para el análisis:



,indicator,horizon,target_col,regime_id,IC_IS,IC_OOS,oos_minus_is,n_pairs_IS,n_pairs_OOS,abs_IC_OOS,note,regime_name,same_sign,abs_gap
0,roc_60,30,t2_dir_thr_30,0,-0.174032,-0.146345,0.027687,107059,88486,0.146345,,overnight,True,0.027687
1,ema_60,30,t2_dir_thr_30,0,-0.158645,-0.138605,0.020040,107059,88486,0.138605,,overnight,True,0.020040
2,roc_30,30,t2_dir_thr_30,0,-0.102924,-0.108434,-0.005510,107059,88486,0.108434,,overnight,True,0.005510
3,bb_60_15,30,t2_dir_thr_30,0,-0.112567,-0.107238,0.005330,107059,88486,0.107238,,overnight,True,0.005330
4,bb_60_20,30,t2_dir_thr_30,0,-0.112567,-0.107238,0.005330,107059,88486,0.107238,,overnight,True,0.005330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,rsi_14,120,t2_dir_thr_120,3,-0.188447,-0.198804,-0.010357,149599,123646,0.198804,,regular,True,0.010357
156,ema_30,120,t2_dir_thr_120,3,-0.186543,-0.192240,-0.005697,149599,123646,0.192240,,regular,True,0.005697
157,roc_30,120,t2_dir_thr_120,3,-0.182862,-0.188120,-0.005258,149599,123646,0.188120,,regular,True,0.005258
158,stoch_k_30,120,t2_dir_thr_120,3,-0.168773,-0.177457,-0.008684,149599,123646,0.177457,,regular,True,0.008684


## **7.3. Conclusiones — Análisis por régimen de mercado (targets T2)**


El análisis por régimen de mercado permite distinguir entre factores estructurales, que mantienen señal en distintos contextos, y factores contextuales, cuya relevancia depende de condiciones específicas del mercado.

En primer lugar, el resumen por régimen confirma que la magnitud de la señal varía significativamente según el contexto intradía. El régimen de apertura (opening) presenta consistentemente los mayores valores de |IC_OOS| en todos los horizontes, alcanzando niveles cercanos a 0.27–0.28. Esto lo posiciona como el entorno donde la señal es más fuerte y explotable. Le sigue el premarket, con valores intermedios, mientras que los regímenes regular y overnight presentan señales más moderadas.

En todos los casos, el porcentaje de consistencia de signo es del 100%, y los valores de abs_gap son bajos. Esto indica que la señal no solo es fuerte en ciertos regímenes, sino también estable fuera de muestra, lo que refuerza su validez para modelado.

Al analizar los factores estructurales, se identifica un conjunto claro de indicadores que aparecen de forma consistente en múltiples regímenes y horizontes. Entre los más relevantes destacan:

- roc_60  
- ema_60  
- roc_30  
- ema_30  
- rsi_14  
- bb_60_15 / bb_60_20  

Estos indicadores están presentes en los cuatro regímenes principales (opening, premarket, regular, overnight) y en todos los horizontes analizados. Además, presentan valores elevados de |IC_OOS| y bajos niveles de abs_gap, lo que los convierte en los factores más robustos del sistema.

Este grupo define el núcleo estructural de la señal. Su presencia transversal indica que capturan dinámicas fundamentales del mercado, principalmente relacionadas con momentum y tendencia. La repetición de indicadores de la misma familia confirma también la existencia de redundancia, lo que sugiere que no es necesario conservar todos ellos en etapas posteriores.

Por otro lado, el análisis de factores contextuales permite identificar indicadores cuya relevancia depende del régimen de mercado. Entre los casos más destacados:

- indicadores de volatilidad (atr_norm_*) aparecen únicamente en el régimen regular, con IC positivos, lo que sugiere que capturan la probabilidad de movimientos significativos más que la dirección,
- algunos indicadores de menor escala (como bb_30_* o stoch_k_20) aparecen en premarket o combinaciones específicas de regímenes,
- roc_20 aparece de forma aislada en el régimen opening para horizontes largos, indicando un comportamiento particular en ese contexto.

Estos factores contextuales pueden aportar valor adicional, pero su uso debe ser más cuidadoso, ya que no generalizan a todos los regímenes.

Otro aspecto relevante es que la estructura de factores se mantiene prácticamente constante entre horizontes. Los mismos indicadores dominan en 30, 60, 90 y 120 minutos, lo que indica que la señal es estructural y no depende críticamente del horizonte temporal. Esto refuerza la idea de que el régimen de mercado es el principal determinante del comportamiento predictivo.

En conjunto, este análisis permite concluir que:

- la señal está fuertemente condicionada por el régimen de mercado, siendo el régimen de apertura el más relevante,
- existe un núcleo de indicadores estructurales robustos que mantienen señal en todos los regímenes,
- la mayor parte de la señal está explicada por factores de momentum y tendencia,
- existe redundancia significativa entre indicadores de la misma familia,
- algunos indicadores aportan información adicional en contextos específicos, actuando como factores contextuales,
- y la estabilidad de la señal entre IS y OOS es alta en todos los regímenes.

Estos resultados consolidan la base para la siguiente etapa, que consiste en seleccionar un subconjunto reducido de indicadores no redundantes, priorizando los factores estructurales y evaluando de forma cuidadosa la inclusión de factores contextuales.

# **8. Ranking de indicadores**


En este punto se priorizan los indicadores técnicos en función de tres criterios clave:

- Magnitud de la señal fuera de muestra (IC_OOS)
- Estabilidad entre IS y OOS (gap bajo)
- Consistencia de signo

El objetivo es identificar los factores más relevantes y confiables para el modelado.

**Criterios de ranking**

Un indicador se considera de alta calidad cuando:

- presenta un |IC_OOS| elevado → señal fuerte
- tiene un gap IS vs OOS bajo → buena generalización
- mantiene el mismo signo → estabilidad direccional

La combinación de estos tres criterios permite filtrar indicadores robustos y evitar señales espurias.

----

Resultados principales
Indicadores mejor rankeados

Los indicadores que dominan el ranking son:

ema_60
roc_60
roc_30
ema_30
bb_60_*
rsi_14
Estos presentan:

alta magnitud de IC_OOS (~0.30 – 0.40)
bajo gap (~0.02)
consistencia de signo en todos los casos
Dominancia de factores de tendencia y momentum

Los indicadores mejor posicionados pertenecen principalmente a:

medias móviles (EMA)
rate of change (ROC)
bandas de Bollinger
RSI
Esto indica que la señal está dominada por dinámicas de:

momentum suavizado
reversión a la media
Indicadores secundarios

Otros indicadores relevantes, pero con menor consistencia o menor cobertura entre regímenes:

stoch_k_*
ema_20
roc_20
Estos pueden aportar valor adicional, pero no constituyen el núcleo del sistema.

Lectura estructural
El ranking es consistente entre horizontes (delta_60 y delta_90)
Los mismos indicadores aparecen en las primeras posiciones
No hay dependencia fuerte de un único régimen
Esto refuerza la idea de que existen factores estructurales dominantes.

Conclusión del ranking
Se identifica un conjunto reducido de indicadores que concentran la mayor parte de la señal

Estos indicadores son:

fuertes en OOS
estables
consistentes
→ Constituyen los principales candidatos para feature selection

Si quieres, el siguiente paso es dejar esto en una lista final de features seleccionadas para pasar al modelado.

## **8.1. Códigos de ranking**

### **1. Ranking global de indicadores**

In [56]:
ranking_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
        pct_same_sign=("same_sign", "mean"),
    )
    .reset_index()
)

ranking_df["pct_same_sign"] = ranking_df["pct_same_sign"] * 100

ranking_df = ranking_df.sort_values(
    ["mean_abs_IC_OOS", "n_regimes"],
    ascending=[False, False]
).reset_index(drop=True)



### **2. Top indicadores “core” (estructurales)**

In [57]:
core_indicators = ranking_df[
    ranking_df["n_regimes"] >= 4
].sort_values("mean_abs_IC_OOS", ascending=False)



### **3. Indicadores secundarios**

In [58]:
secondary_indicators = ranking_df[
    (ranking_df["n_regimes"] >= 2) &
    (ranking_df["n_regimes"] < 4)
].sort_values("mean_abs_IC_OOS", ascending=False)



### **4. Indicadores contextuales**

In [59]:
contextual_indicators = ranking_df[
    ranking_df["n_regimes"] == 1
].sort_values("mean_abs_IC_OOS", ascending=False)



### **5. Validación de estabilidad**

In [60]:
ranking_df[[
    "indicator",
    "mean_abs_IC_OOS",
    "mean_abs_gap",
    "pct_same_sign"
]].sort_values("mean_abs_IC_OOS", ascending=False).head(10)

,indicator,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,roc_60,0.273160,0.022389,100.0
1,roc_20,0.268645,0.020978,100.0
2,ema_60,0.260113,0.025191,100.0
3,stoch_k_30,0.231213,0.015529,100.0
4,ema_20,0.220789,0.019020,100.0
5,roc_30,0.219772,0.027260,100.0
6,ema_30,0.209328,0.022590,100.0
7,bb_60_15,0.200450,0.023556,100.0
8,bb_60_20,0.200450,0.023556,100.0
9,rsi_14,0.196704,0.019974,100.0


### **6. Clasificación por familia (para justificar momentum / mean reversion)**

In [61]:
def classify_family(indicator: str) -> str:
    s = indicator.lower()
    if s.startswith("ema_"): return "EMA"
    if s.startswith("roc_"): return "ROC"
    if s.startswith("rsi_"): return "RSI"
    if s.startswith("bb_"): return "BB"
    if s.startswith("stoch_"): return "STOCH"
    return "OTHER"

ranking_df["family"] = ranking_df["indicator"].apply(classify_family)

family_summary = (
    ranking_df.groupby("family")
    .agg(
        n_indicators=("indicator", "count"),
        mean_abs_IC_OOS=("mean_abs_IC_OOS", "mean")
    )
    .sort_values("mean_abs_IC_OOS", ascending=False)
)




## **8.2. Resultados**

In [62]:
print('\n Ranking global de indicadores: \n')
display (ranking_df.head(15))

print('\n Top indicadores “core” (estructurales): \n')
display (core_indicators)


print('\n Indicadores secundarios')
display(secondary_indicators)

print('\n Indicadores contextuales')
display(contextual_indicators)


print('\n Clasificación por familia (para justificar momentum / mean reversion)')
display(family_summary)


print('\nValidación de estabilidad')
ranking_df[[
    "indicator",
    "mean_abs_IC_OOS",
    "mean_abs_gap",
    "pct_same_sign"
]].sort_values("mean_abs_IC_OOS", ascending=False).head(10)



 Ranking global de indicadores: 



,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,family
0,roc_60,4,-0.273160,0.273160,0.022389,100.0,ROC
1,roc_20,2,-0.268645,0.268645,0.020978,100.0,ROC
2,ema_60,4,-0.260113,0.260113,0.025191,100.0,EMA
3,stoch_k_30,4,-0.231213,0.231213,0.015529,100.0,STOCH
4,ema_20,3,-0.220789,0.220789,0.019020,100.0,EMA
5,roc_30,4,-0.219772,0.219772,0.027260,100.0,ROC
6,ema_30,4,-0.209328,0.209328,0.022590,100.0,EMA
7,bb_60_15,4,-0.200450,0.200450,0.023556,100.0,BB
8,bb_60_20,4,-0.200450,0.200450,0.023556,100.0,BB
9,rsi_14,4,-0.196704,0.196704,0.019974,100.0,RSI



 Top indicadores “core” (estructurales): 



,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,roc_60,4,-0.273160,0.273160,0.022389,100.0
2,ema_60,4,-0.260113,0.260113,0.025191,100.0
3,stoch_k_30,4,-0.231213,0.231213,0.015529,100.0
5,roc_30,4,-0.219772,0.219772,0.027260,100.0
6,ema_30,4,-0.209328,0.209328,0.022590,100.0
7,bb_60_15,4,-0.200450,0.200450,0.023556,100.0
8,bb_60_20,4,-0.200450,0.200450,0.023556,100.0
9,rsi_14,4,-0.196704,0.196704,0.019974,100.0



 Indicadores secundarios


...

,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
11,stoch_k_20,1,-0.141257,0.141257,0.032098,100.0
13,bb_30_25,1,-0.135569,0.135569,0.024701,100.0
15,atr_norm_10,1,0.120736,0.120736,0.003377,100.0
16,atr_norm_14,1,0.108121,0.108121,0.001903,100.0
17,atr_norm_30,1,0.097938,0.097938,0.003460,100.0
18,atr_norm_20,1,0.096469,0.096469,0.002378,100.0



 Clasificación por familia (para justificar momentum / mean reversion)


,n_indicators,mean_abs_IC_OOS
family,,
ROC,3,0.253859
EMA,3,0.230077
RSI,1,0.196704
STOCH,2,0.186235
BB,6,0.157715
OTHER,4,0.105816



Validación de estabilidad


,indicator,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,roc_60,0.273160,0.022389,100.0
1,roc_20,0.268645,0.020978,100.0
2,ema_60,0.260113,0.025191,100.0
3,stoch_k_30,0.231213,0.015529,100.0
4,ema_20,0.220789,0.019020,100.0
5,roc_30,0.219772,0.027260,100.0
6,ema_30,0.209328,0.022590,100.0
7,bb_60_15,0.200450,0.023556,100.0
8,bb_60_20,0.200450,0.023556,100.0
9,rsi_14,0.196704,0.019974,100.0


## **8.3. Conclusiones del ranking**

El ranking global de indicadores confirma de forma consistente la existencia de un conjunto reducido de factores que concentran la mayor parte de la señal del sistema. Estos resultados permiten separar claramente entre indicadores estructurales, secundarios y contextuales.

**Indicadores core (estructurales)**

Los indicadores que aparecen en los cuatro regímenes y dominan el ranking son:

- roc_60  
- ema_60  
- stoch_k_30  
- roc_30  
- ema_30  
- bb_60_15 / bb_60_20  
- rsi_14  

Estos indicadores presentan simultáneamente:

- alta magnitud de señal (|IC_OOS| ≈ 0.20 – 0.27)  
- bajo gap entre IS y OOS (≈ 0.02)  
- consistencia de signo total (100%)  
- presencia en todos los regímenes  

Esto los convierte en el núcleo estructural del sistema. Representan factores robustos, generalizables y con capacidad predictiva estable.

**Dominancia por familias**

El análisis por familias refuerza una conclusión clave:

- ROC (momentum) es la familia más fuerte  
- EMA (tendencia) es la segunda más relevante  
- RSI y STOCH aportan señal adicional  
- Bollinger Bands capturan desviaciones y mean reversion  

Esto indica que la señal del sistema está dominada por:

- momentum  
- tendencia  
- reversión a la media  

Es decir, el comportamiento del mercado intradía está bien explicado por dinámicas de continuidad y reversión controlada.

**Indicadores secundarios**

Un segundo grupo de indicadores presenta buena señal, pero menor robustez o menor cobertura entre regímenes:

- roc_20  
- ema_20  
- bb_60_25  
- bb_30_15  
- bb_30_20  

Estos indicadores:

- mantienen buena magnitud de IC  
- son estables  
- pero no aparecen en todos los regímenes  

Por lo tanto, pueden aportar valor adicional, pero no son esenciales. Su inclusión debe evaluarse en función de la redundancia.

**Indicadores contextuales**

Algunos indicadores aparecen únicamente en un régimen específico:

- stoch_k_20  
- bb_30_25  
- atr_norm_* (volatilidad)  

En particular, los indicadores de volatilidad (ATR normalizado) presentan:

- IC positivo  
- muy bajo gap  
- alta estabilidad  

Esto sugiere que no predicen dirección, sino probabilidad de movimiento significativo, lo cual es coherente con la definición del target T2.

Estos indicadores deben considerarse como factores contextuales, útiles en escenarios específicos, pero no generalizables.

**Redundancia entre indicadores**

El ranking muestra claramente que existen múltiples indicadores de la misma familia con comportamiento casi idéntico:

- bb_60_15, bb_60_20, bb_60_25  
- roc_60, roc_30, roc_20  
- ema_60, ema_30, ema_20  

Esto implica que el sistema contiene redundancia significativa. Muchos indicadores están capturando la misma información, por lo que no es necesario utilizarlos todos.

**Estabilidad de la señal**

Todos los indicadores mejor posicionados presentan:

- pct_same_sign = 100%  
- gap bajo  
- IC_OOS consistente  

Esto confirma que la señal:

- no es ruido  
- no está sobreajustada  
- es estable fuera de muestra  

**Conclusión final del ranking**

El análisis permite identificar un conjunto compacto de indicadores que concentran la mayor parte del edge del sistema.

Este conjunto:

- es consistente entre regímenes  
- es estable entre IS y OOS  
- presenta alta magnitud de señal  
- está dominado por momentum y tendencia  

Por lo tanto, estos indicadores constituyen la base para la siguiente etapa: definir un subconjunto final de features no redundantes que será utilizado en el modelado.

# **9. Análisis de repetición entre regímenes**
   



   Se identifican indicadores que:

* aparecen consistentemente en varios regímenes

Objetivo:
Detectar factores robustos globales.

## **9.1. Código**

### **1. Análisis de repetición entre regímenes**

In [63]:
# ============================================================
# 1) FRECUENCIA DE APARICIÓN POR RÉGIMEN
# ============================================================

repetition_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        regimes=("regime_name", lambda x: sorted(x.unique())),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
    )
    .reset_index()
    .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
)



### **2. Indicadores globales**

In [64]:
global_factors = repetition_df[
    repetition_df["n_regimes"] >= 4
]


### **3. Indicadores semi-globales**

In [65]:
semi_global_factors = repetition_df[
    (repetition_df["n_regimes"] == 3)
]


### **4. Indicadores débiles / locales**

In [66]:
local_factors = repetition_df[
    repetition_df["n_regimes"] <= 2
]


### **5. Validación: relación repetición vs señal**

In [67]:
repetition_df[[
    "indicator",
    "n_regimes",
    "mean_abs_IC_OOS",
    "mean_abs_gap"
]].sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])

,indicator,n_regimes,mean_abs_IC_OOS,mean_abs_gap
15,roc_60,4,0.273160,0.022389
12,ema_60,4,0.260113,0.025191
18,stoch_k_30,4,0.231213,0.015529
14,roc_30,4,0.219772,0.027260
11,ema_30,4,0.209328,0.022590
7,bb_60_15,4,0.200450,0.023556
8,bb_60_20,4,0.200450,0.023556
16,rsi_14,4,0.196704,0.019974
10,ema_20,3,0.220789,0.019020
9,bb_60_25,3,0.146625,0.023344


## **9.2. Resultados**

In [68]:

print('\n1. Análisis de repetición entre regímenes')
display(repetition_df)

print('\n2. Indicadores globales')
display(global_factors)

print('\n3. Indicadores semi-globales')
display(semi_global_factors)

print('\n4. Indicadores débiles / locales')
display(local_factors)

print('\n5. Validación: relación repetición vs señal')
repetition_df[[
    "indicator",
    "n_regimes",
    "mean_abs_IC_OOS",
    "mean_abs_gap"
]].sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])


1. Análisis de repetición entre regímenes


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
15,roc_60,4,"[opening, overnight, premarket, regular]",-0.273160,0.273160,0.022389
12,ema_60,4,"[opening, overnight, premarket, regular]",-0.260113,0.260113,0.025191
18,stoch_k_30,4,"[opening, overnight, premarket, regular]",-0.231213,0.231213,0.015529
14,roc_30,4,"[opening, overnight, premarket, regular]",-0.219772,0.219772,0.027260
11,ema_30,4,"[opening, overnight, premarket, regular]",-0.209328,0.209328,0.022590
7,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.023556
8,bb_60_20,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.023556
16,rsi_14,4,"[opening, overnight, premarket, regular]",-0.196704,0.196704,0.019974
10,ema_20,3,"[opening, overnight, premarket]",-0.220789,0.220789,0.019020
9,bb_60_25,3,"[overnight, premarket, regular]",-0.146625,0.146625,0.023344



2. Indicadores globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
15,roc_60,4,"[opening, overnight, premarket, regular]",-0.273160,0.273160,0.022389
12,ema_60,4,"[opening, overnight, premarket, regular]",-0.260113,0.260113,0.025191
18,stoch_k_30,4,"[opening, overnight, premarket, regular]",-0.231213,0.231213,0.015529
14,roc_30,4,"[opening, overnight, premarket, regular]",-0.219772,0.219772,0.027260
11,ema_30,4,"[opening, overnight, premarket, regular]",-0.209328,0.209328,0.022590
7,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.023556
8,bb_60_20,4,"[opening, overnight, premarket, regular]",-0.200450,0.200450,0.023556
16,rsi_14,4,"[opening, overnight, premarket, regular]",-0.196704,0.196704,0.019974



3. Indicadores semi-globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
10,ema_20,3,"[opening, overnight, premarket]",-0.220789,0.220789,0.019020
9,bb_60_25,3,"[overnight, premarket, regular]",-0.146625,0.146625,0.023344
4,bb_30_15,3,"[overnight, premarket, regular]",-0.139760,0.139760,0.019776



4. Indicadores débiles / locales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
13,roc_20,2,"[opening, overnight]",-0.268645,0.268645,0.020978
5,bb_30_20,2,"[overnight, premarket]",-0.123434,0.123434,0.024448
17,stoch_k_20,1,[premarket],-0.141257,0.141257,0.032098
6,bb_30_25,1,[premarket],-0.135569,0.135569,0.024701
0,atr_norm_10,1,[regular],0.120736,0.120736,0.003377
1,atr_norm_14,1,[regular],0.108121,0.108121,0.001903
3,atr_norm_30,1,[regular],0.097938,0.097938,0.003460
2,atr_norm_20,1,[regular],0.096469,0.096469,0.002378



5. Validación: relación repetición vs señal


,indicator,n_regimes,mean_abs_IC_OOS,mean_abs_gap
15,roc_60,4,0.273160,0.022389
12,ema_60,4,0.260113,0.025191
18,stoch_k_30,4,0.231213,0.015529
14,roc_30,4,0.219772,0.027260
11,ema_30,4,0.209328,0.022590
7,bb_60_15,4,0.200450,0.023556
8,bb_60_20,4,0.200450,0.023556
16,rsi_14,4,0.196704,0.019974
10,ema_20,3,0.220789,0.019020
9,bb_60_25,3,0.146625,0.023344


## **9.3. Análisis de resultados**


**1. Existen indicadores claramente globales**

* Los más importantes son:

  * `ema_60`, `roc_60`
  * `ema_30`, `roc_30`
  * `bb_60_15`, `rsi_14`

* Aparecen en **todos los regímenes**
  → Son **robustos y confiables**

---

**2. La señal principal es estructural**

* Los indicadores que más se repiten:

  * también son los que tienen mayor IC

→ La señal **no depende del régimen**, es general del mercado

---

**3. Más repetición = mejor indicador**

* A mayor `n_regimes`:

  * mayor consistencia
  * mayor estabilidad
  * mejor desempeño

→ La repetición es un **criterio clave de calidad**

---

**4. Existen indicadores útiles pero no globales**

* Ejemplo:

  * `ema_20`, `stoch_k_30`, `bb_60_20`

* Funcionan bien, pero:

  * no en todos los regímenes

→ Son **complementarios**, no principales

---

**5. Algunos indicadores son solo contextuales**

* Ejemplo:

  * `bb_30_20`, `bb_30_25` (solo premarket)

→ No generalizan
→ Dependen del contexto

---

**6. Importante: señal alta no implica robustez**

* Ejemplo:

  * `stoch_k_20`, `roc_20` tienen IC alto

Pero:

* aparecen en pocos regímenes

→ Son **fuertes pero poco robustos**

---

**Conclusión final del punto 9**

* Los mejores indicadores son:

  * **los que se repiten en todos los regímenes**
  * **no los que tienen solo IC alto**

→ La señal del problema es **global y estructural**, no específica de un régimen


# **10. Identificación de indicadores específicos por régimen**

Se buscan factores que:

* funcionan bien en un régimen
* no en otros

Objetivo:
Capturar señales contextuales.

## **10.1. Código**

In [ ]:
import pandas as pd

def find_regime_specific_indicators(
    robustness_top_all: pd.DataFrame,
    *,
    top_n_per_regime: int = 10,
) -> pd.DataFrame:
    """
    Identifica indicadores específicos por régimen:
    aparecen en el top de un régimen y no aparecen
    en el top de ningún otro régimen.

    Parámetros
    ----------
    robustness_top_all : pd.DataFrame
        DataFrame combinado con los top robustos de todos los regímenes.
        Debe contener al menos:
        - regime_name
        - indicator
        - IC_OOS
        - abs_IC_OOS
        - abs_gap

    top_n_per_regime : int
        Cantidad de indicadores top a considerar por cada régimen.

    Devuelve
    --------
    pd.DataFrame
        Tabla con indicadores específicos por régimen.
        Puede devolver DataFrame vacío, lo cual significa que
        no hubo indicadores exclusivos.
    """
    required_cols = {"regime_name", "indicator", "IC_OOS", "abs_IC_OOS", "abs_gap"}
    missing = required_cols - set(robustness_top_all.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

    # 1) Tomar top N por régimen
    top_by_regime = (
        robustness_top_all
        .sort_values(["regime_name", "abs_IC_OOS"], ascending=[True, False])
        .groupby("regime_name", group_keys=False)
        .head(top_n_per_regime)
        .copy()
    )

    # 2) Construir sets de indicadores por régimen
    sets_by_regime = {
        regime: set(g["indicator"].tolist())
        for regime, g in top_by_regime.groupby("regime_name")
    }

    # 3) Buscar exclusivos
    rows = []

    for regime, current_set in sets_by_regime.items():
        other_sets = [s for r, s in sets_by_regime.items() if r != regime]
        others_union = set().union(*other_sets) if other_sets else set()

        specific_indicators = sorted(current_set - others_union)

        if not specific_indicators:
            continue

        df_reg = top_by_regime[top_by_regime["regime_name"] == regime].copy()

        df_reg = df_reg[df_reg["indicator"].isin(specific_indicators)].copy()

        for _, row in df_reg.iterrows():
            rows.append({
                "regime": regime,
                "indicator": row["indicator"],
                "IC_OOS": row["IC_OOS"],
                "abs_IC_OOS": row["abs_IC_OOS"],
                "abs_gap": row["abs_gap"],
            })

    # 4) Armar salida robusta
    if len(rows) == 0:
        return pd.DataFrame(
            columns=["regime", "indicator", "IC_OOS", "abs_IC_OOS", "abs_gap"]
        )

    out = pd.DataFrame(rows).sort_values(
        ["regime", "abs_IC_OOS"],
        ascending=[True, False]
    ).reset_index(drop=True)

    return out

In [ ]:
specific_by_regime = find_regime_specific_indicators(
    robustness_top_all,
    top_n_per_regime=10,
)



##**10.2. Resultados**

In [ ]:
specific_by_regime

,regime,indicator,IC_OOS,abs_IC_OOS,abs_gap


## **10.3. Observaciones**

**1. No existen indicadores exclusivos por régimen**

* Ningún indicador aparece solo en un régimen
  → Todos los relevantes se repiten en varios

---

**2. La señal no es contextual**

* No hay factores que funcionen solo en:

  * premarket
  * opening
  * overnight
  * etc.

→ El comportamiento del mercado es **consistente entre regímenes**

---

**3. Predominan los factores globales**

* Los mismos indicadores dominan en todos los contextos:

  * EMA
  * ROC
  * RSI
  * BB

→ La señal es **estructural, no dependiente del régimen**

---

**4. No es necesario modelar por régimen**

* Dado que no hay diferencias claras:

  * no hace falta separar modelos por régimen
  * ni features específicas por régimen

---

**5. Simplificación del modelo**

* Se puede usar un único set de features
* sin necesidad de lógica condicional por régimen

---
**Conclusión final del punto 10**

* No hay evidencia de señales específicas por régimen
* El problema está dominado por **factores globales y robustos**

→ El modelo debe enfocarse en **features generales, no contextuales**


# **11. Análisis de redundancia (correlación entre indicadores)**

Se evalúa:

* correlación entre features
* agrupación de indicadores similares

Objetivo:

* eliminar duplicados
* reducir dimensionalidad
* evitar multicolinealidad


## **11.1. Código**

### **1) Seleccionar indicadores relevantes (los que ya validaste)**

In [69]:
import numpy as np
import pandas as pd

def analyze_all_feature_redundancy(
    df: pd.DataFrame,
    *,
    indicator_columns: list[str] | None = None,
    corr_threshold: float = 0.90,
    ranking_df: pd.DataFrame | None = None,
    print_report: bool = True,
) -> dict:
    """
    Analiza redundancia entre TODOS los indicadores técnicos.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset con indicadores técnicos.
    indicator_columns : list[str] | None
        Lista explícita de indicadores a evaluar.
        Si es None, se infiere automáticamente.
    corr_threshold : float
        Umbral de correlación absoluta para considerar redundancia.
    ranking_df : pd.DataFrame | None
        Opcional. Si se pasa, se usa mean_abs_IC_OOS para elegir
        el mejor representante dentro de cada grupo redundante.
        Debe tener columnas: indicator, mean_abs_IC_OOS
    print_report : bool
        Si True, imprime resúmenes.

    Devuelve
    --------
    dict con:
    - feature_list
    - family_summary_df
    - corr_matrix
    - high_corr_pairs_df
    - redundancy_groups_df
    - representatives_df
    """

    # ============================================================
    # 1) Definir TODOS los indicadores a evaluar
    # ============================================================
    if indicator_columns is None:
        exclude_cols = {
            "date", "minute_of_day",
            "open", "high", "low", "close", "volume",
            "delta_60", "delta_90",
            "ret_60", "ret_90",
            "split_fe",
            "is_overnight", "is_premarket", "is_opening", "is_regular", "is_closing",
            "is_mon", "is_tue", "is_wed", "is_thu", "is_fri",
        }

        indicator_columns = [
            c for c in df.columns
            if c not in exclude_cols
        ]

    # conservar solo columnas existentes y numéricas
    indicator_columns = [
        c for c in indicator_columns
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
    ]

    if len(indicator_columns) == 0:
        raise ValueError("No se encontraron indicadores técnicos válidos para analizar.")

    # ============================================================
    # 2) Clasificación por familia
    # ============================================================
    def classify_family(indicator: str) -> str:
        s = indicator.lower()
        if s.startswith("ema_"):
            return "EMA"
        if s.startswith("roc_"):
            return "ROC"
        if s.startswith("rsi_"):
            return "RSI"
        if s.startswith("bb_"):
            return "BB"
        if s.startswith("stoch_"):
            return "STOCH"
        if s.startswith("atr_"):
            return "ATR"
        if s.startswith("mom_"):
            return "MOM"
        if "volume" in s:
            return "VOLUME"
        if "macd" in s:
            return "MACD"
        return "OTHER"

    feature_df = pd.DataFrame({
        "indicator": indicator_columns,
        "family": [classify_family(c) for c in indicator_columns],
    })

    family_summary_df = (
        feature_df.groupby("family", as_index=False)
        .agg(
            n_indicators=("indicator", "count"),
            indicators=("indicator", lambda s: sorted(s.tolist()))
        )
        .sort_values(["n_indicators", "family"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # ============================================================
    # 3) Matriz de correlación
    # ============================================================
    corr_matrix = df[indicator_columns].corr()

    # ============================================================
    # 4) Pares altamente correlacionados
    # ============================================================
    high_corr_rows = []

    for i in range(len(indicator_columns)):
        for j in range(i + 1, len(indicator_columns)):
            f1 = indicator_columns[i]
            f2 = indicator_columns[j]
            corr = corr_matrix.loc[f1, f2]

            if pd.notna(corr) and abs(corr) >= corr_threshold:
                high_corr_rows.append({
                    "feature_1": f1,
                    "family_1": classify_family(f1),
                    "feature_2": f2,
                    "family_2": classify_family(f2),
                    "correlation": corr,
                    "abs_correlation": abs(corr),
                })

    high_corr_pairs_df = pd.DataFrame(high_corr_rows)

    if not high_corr_pairs_df.empty:
        high_corr_pairs_df = high_corr_pairs_df.sort_values(
            ["abs_correlation", "feature_1", "feature_2"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

    # ============================================================
    # 5) Grupos de redundancia (componentes conectadas)
    # ============================================================
    adjacency = {f: set() for f in indicator_columns}

    if not high_corr_pairs_df.empty:
        for _, row in high_corr_pairs_df.iterrows():
            f1 = row["feature_1"]
            f2 = row["feature_2"]
            adjacency[f1].add(f2)
            adjacency[f2].add(f1)

    visited = set()
    groups = []

    for f in indicator_columns:
        if f in visited:
            continue

        stack = [f]
        component = []

        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            component.append(node)

            for neigh in adjacency[node]:
                if neigh not in visited:
                    stack.append(neigh)

        groups.append(sorted(component))

    redundancy_rows = []
    for g_id, group in enumerate(groups, start=1):
        redundancy_rows.append({
            "group_id": g_id,
            "n_features": len(group),
            "features": group,
            "families": sorted(set(classify_family(x) for x in group)),
        })

    redundancy_groups_df = (
        pd.DataFrame(redundancy_rows)
        .sort_values(["n_features", "group_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # ============================================================
    # 6) Elegir representante por grupo
    # ============================================================
    ranking_map = {}
    if ranking_df is not None:
        if {"indicator", "mean_abs_IC_OOS"}.issubset(ranking_df.columns):
            ranking_map = dict(
                zip(ranking_df["indicator"], ranking_df["mean_abs_IC_OOS"])
            )

    rep_rows = []
    for _, row in redundancy_groups_df.iterrows():
        group = row["features"]

        if len(group) == 1:
            representative = group[0]
            reason = "único en el grupo"
        else:
            if ranking_map:
                representative = max(group, key=lambda x: ranking_map.get(x, -np.inf))
                reason = "mayor mean_abs_IC_OOS dentro del grupo"
            else:
                representative = group[0]
                reason = "primer indicador del grupo (sin ranking externo)"

        rep_rows.append({
            "group_id": row["group_id"],
            "representative": representative,
            "n_features_in_group": row["n_features"],
            "group_features": group,
            "reason": reason,
            "representative_family": classify_family(representative),
            "representative_mean_abs_IC_OOS": ranking_map.get(representative, np.nan),
        })

    representatives_df = pd.DataFrame(rep_rows)

    # ============================================================
    # 7) Reporte
    # ============================================================
    if print_report:
        print("=" * 100)
        print("ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS")
        print("=" * 100)
        print(f"Cantidad total de features evaluadas: {len(indicator_columns)}")
        print(f"Umbral de correlación absoluta: {corr_threshold:.2f}")
        print(f"Pares altamente correlacionados encontrados: {len(high_corr_pairs_df)}")
        print(f"Cantidad de grupos de redundancia: {len(redundancy_groups_df)}")
        print()

        print("-" * 100)
        print("LISTADO TOTAL DE FEATURES EVALUADAS")
        print("-" * 100)
        print(indicator_columns)
        print()

        print("-" * 100)
        print("RESUMEN POR FAMILIA")
        print("-" * 100)
        print(family_summary_df[["family", "n_indicators"]].to_string(index=False))
        print()

        if not high_corr_pairs_df.empty:
            print("-" * 100)
            print("TOP PARES ALTAMENTE CORRELACIONADOS")
            print("-" * 100)
            print(
                high_corr_pairs_df.head(20)[
                    ["feature_1", "feature_2", "correlation", "abs_correlation"]
                ].to_string(index=False)
            )
            print()
        else:
            print("[OK] No se encontraron pares con correlación superior al umbral.")
            print()

        print("-" * 100)
        print("GRUPOS DE REDUNDANCIA")
        print("-" * 100)
        print(
            redundancy_groups_df[["group_id", "n_features", "families", "features"]]
            .head(20)
            .to_string(index=False)
        )
        print()

        print("-" * 100)
        print("REPRESENTANTES PROPUESTOS POR GRUPO")
        print("-" * 100)
        print(
            representatives_df[
                ["group_id", "representative", "representative_family", "n_features_in_group", "reason"]
            ].to_string(index=False)
        )

    return {
        "feature_list": indicator_columns,
        "family_summary_df": family_summary_df,
        "corr_matrix": corr_matrix,
        "high_corr_pairs_df": high_corr_pairs_df,
        "redundancy_groups_df": redundancy_groups_df,
        "representatives_df": representatives_df,
    }

### **2) Sin ranking externo**

In [70]:
redundancy_results = analyze_all_feature_redundancy(
    mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    corr_threshold=0.90,
    print_report=True,
)

ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS
Cantidad total de features evaluadas: 42
Umbral de correlación absoluta: 0.90
Pares altamente correlacionados encontrados: 100
Cantidad de grupos de redundancia: 9

----------------------------------------------------------------------------------------------------
LISTADO TOTAL DE FEATURES EVALUADAS
----------------------------------------------------------------------------------------------------
['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60']

-----------------

### **3) Usando ranking global para elegir mejor representante por grupo**

In [71]:
redundancy_results = analyze_all_feature_redundancy(
    mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    corr_threshold=0.90,
    ranking_df=ranking_df,
    print_report=True,
)

ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS
Cantidad total de features evaluadas: 42
Umbral de correlación absoluta: 0.90
Pares altamente correlacionados encontrados: 100
Cantidad de grupos de redundancia: 9

----------------------------------------------------------------------------------------------------
LISTADO TOTAL DE FEATURES EVALUADAS
----------------------------------------------------------------------------------------------------
['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60']

-----------------

## **11.2. Resultados**

In [72]:
print('\nListado total de features evaluadas\n')
display(redundancy_results["feature_list"])

print('\nResumen por familia\n')
display(redundancy_results["family_summary_df"])

print('\nPares altamente correlacionados\n')
display(redundancy_results["high_corr_pairs_df"])

print('\nGrupos redundantes\n')
display(redundancy_results["redundancy_groups_df"])

print('\nRepresentantes sugeridos\n')
display(redundancy_results["representatives_df"])


Listado total de features evaluadas



['rsi_14',
 'rsi_7',
 'rsi_5',
 'rsi_3',
 'mom_10',
 'mom_5',
 'mom_3',
 'volume_ratio_15',
 'volume_ratio_20',
 'volume_ratio_30',
 'volume_ratio_60',
 'volume_ratio_90',
 'macd',
 'ema_15',
 'ema_20',
 'ema_30',
 'ema_60',
 'stoch_k_14',
 'stoch_k_20',
 'stoch_k_30',
 'bb_15_15',
 'bb_20_15',
 'bb_30_15',
 'bb_60_15',
 'bb_15_20',
 'bb_20_20',
 'bb_30_20',
 'bb_60_20',
 'bb_15_25',
 'bb_20_25',
 'bb_30_25',
 'bb_60_25',
 'atr_norm_5',
 'atr_norm_10',
 'atr_norm_14',
 'atr_norm_20',
 'atr_norm_30',
 'roc_5',
 'roc_10',
 'roc_20',
 'roc_30',
 'roc_60']


Resumen por familia



,family,n_indicators,indicators
0,BB,12,"[bb_15_15, bb_15_20, bb_15_25, bb_20_15, bb_20..."
1,ATR,5,"[atr_norm_10, atr_norm_14, atr_norm_20, atr_no..."
2,ROC,5,"[roc_10, roc_20, roc_30, roc_5, roc_60]"
3,VOLUME,5,"[volume_ratio_15, volume_ratio_20, volume_rati..."
4,EMA,4,"[ema_15, ema_20, ema_30, ema_60]"
5,RSI,4,"[rsi_14, rsi_3, rsi_5, rsi_7]"
6,MOM,3,"[mom_10, mom_3, mom_5]"
7,STOCH,3,"[stoch_k_14, stoch_k_20, stoch_k_30]"
8,MACD,1,[macd]



Pares altamente correlacionados



,feature_1,family_1,feature_2,family_2,correlation,abs_correlation
0,bb_20_20,BB,bb_20_25,BB,1.000000,1.000000
1,mom_5,MOM,roc_5,ROC,1.000000,1.000000
2,bb_60_15,BB,bb_60_20,BB,1.000000,1.000000
3,bb_20_15,BB,bb_20_20,BB,1.000000,1.000000
4,bb_30_15,BB,bb_30_20,BB,1.000000,1.000000
...,...,...,...,...,...,...
95,rsi_7,RSI,bb_30_20,BB,0.911798,0.911798
96,volume_ratio_30,VOLUME,volume_ratio_90,VOLUME,0.902676,0.902676
97,ema_30,EMA,roc_20,ROC,0.902641,0.902641
98,mom_10,MOM,ema_15,EMA,0.902483,0.902483



Grupos redundantes



,group_id,n_features,features,families
0,1,19,"[bb_15_15, bb_15_20, bb_15_25, bb_20_15, bb_20...","[BB, RSI, STOCH]"
1,2,7,"[ema_15, ema_20, ema_30, ema_60, mom_10, roc_1...","[EMA, MOM, ROC]"
2,5,5,"[volume_ratio_15, volume_ratio_20, volume_rati...",[VOLUME]
3,7,5,"[atr_norm_10, atr_norm_14, atr_norm_20, atr_no...",[ATR]
4,3,2,"[mom_5, roc_5]","[MOM, ROC]"
5,4,1,[mom_3],[MOM]
6,6,1,[macd],[MACD]
7,8,1,[roc_30],[ROC]
8,9,1,[roc_60],[ROC]



Representantes sugeridos



,group_id,representative,n_features_in_group,group_features,reason,representative_family,representative_mean_abs_IC_OOS
0,1,stoch_k_30,19,"[bb_15_15, bb_15_20, bb_15_25, bb_20_15, bb_20...",mayor mean_abs_IC_OOS dentro del grupo,STOCH,0.231213
1,2,roc_20,7,"[ema_15, ema_20, ema_30, ema_60, mom_10, roc_1...",mayor mean_abs_IC_OOS dentro del grupo,ROC,0.268645
2,5,volume_ratio_15,5,"[volume_ratio_15, volume_ratio_20, volume_rati...",mayor mean_abs_IC_OOS dentro del grupo,VOLUME,NaN
3,7,atr_norm_10,5,"[atr_norm_10, atr_norm_14, atr_norm_20, atr_no...",mayor mean_abs_IC_OOS dentro del grupo,ATR,0.120736
4,3,mom_5,2,"[mom_5, roc_5]",mayor mean_abs_IC_OOS dentro del grupo,MOM,NaN
5,4,mom_3,1,[mom_3],único en el grupo,MOM,NaN
6,6,macd,1,[macd],único en el grupo,MACD,NaN
7,8,roc_30,1,[roc_30],único en el grupo,ROC,0.219772
8,9,roc_60,1,[roc_60],único en el grupo,ROC,0.273160


In [73]:
def compute_ic_by_day(df, feature, target):
    ic_list = []

    for date, g in df.groupby("date"):
        if g[feature].notna().sum() > 10:
            ic = g[feature].corr(g[target])
            if pd.notna(ic):
                ic_list.append(ic)

    return np.mean(ic_list)

In [ ]:
results = []


features_to_check = [
    "ema_60",
    "roc_60",
    "roc_30",
    "stoch_k_20",
    "volume_ratio_15",
    "atr_norm_10",
    "macd",
]


for f in features_to_check:
    for target in ["delta_60", "delta_90"]:
        ic = compute_ic_by_day(
            mnq_intraday_with_indicators,
            f,
            target
        )

        results.append({
            "feature": f,
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic)
        })

pd.DataFrame(results).sort_values("abs_IC", ascending=False)

,feature,target,IC,abs_IC
1,ema_60,delta_90,-0.194155,0.194155
3,roc_60,delta_90,-0.182031,0.182031
0,ema_60,delta_60,-0.156550,0.156550
2,roc_60,delta_60,-0.144045,0.144045
5,roc_30,delta_90,-0.139204,0.139204
4,roc_30,delta_60,-0.110168,0.110168
11,atr_norm_10,delta_90,0.102836,0.102836
10,atr_norm_10,delta_60,0.090375,0.090375
7,stoch_k_20,delta_90,-0.089873,0.089873
6,stoch_k_20,delta_60,-0.071651,0.071651


## **11.3. Análisis de resultados**

**1. Sí hay señal (confirmado)**

* Los mejores IC están en:

  * `ema_60` → ~0.19
  * `roc_60` → ~0.18
  * `roc_30` → ~0.14

- Esto confirma:

  * señal real
  * consistente con lo visto antes

---

##**2. Ranking real de features**

- Fuertes (core)

  * `ema_60`
  * `roc_60`
  * `roc_30`

  Estos son los verdaderamente importantes

---

- Moderados

  * `atr_norm_10` (~0.10)
  * `stoch_k_20` (~0.07–0.09)

  aportan, pero menos

---

- Débiles / ruido

  * `macd` (~0.02)
  * `volume_ratio_15` (~0.00)

  no aportan señal útil

---

**3. Insight clave**

- La señal está concentrada en:

  * **tendencia (EMA)**
  * **momentum (ROC)**

  Todo lo demás es secundario

---

**4. Importante: signo negativo**

- Todos los buenos son negativos:

  * EMA ↑ → retorno futuro ↓
  * ROC ↑ → retorno futuro ↓

- Interpretación:

  * comportamiento **mean-reverting**

---

**5. Feature set final (limpio y validado)**

- Recomendado

  ```python
  final_features = [
      "ema_60",
      "roc_60",
      "roc_30",
      "atr_norm_10",   # opcional
      "stoch_k_20",    # opcional
  ]
  ```

---

- Eliminar

  * `macd`
  * `volume_ratio_*`

---

**6. Conclusión final del punto 11**

- Ya tienes:

  * features con señal (IC)
  * sin redundancia crítica
  * y con interpretación clara

- El problema está dominado por:

  * tendencia + momentum (mean reversion)

---

**Conclusión final (importante)**

Punto clave del proyecto:

- **La señal existe, pero es moderada (~0.10–0.20)**
- suficiente para modelado, pero no trivial

# **12. Selección final de features**

**1. Criterios de selección**

La selección de variables se realizó en base a los siguientes criterios:

- Señal fuera de muestra (IC OOS)
- Robustez (consistencia entre IS y OOS)
- Consistencia de signo
- Baja redundancia entre indicadores

El objetivo es definir un conjunto de features que capture la señal real del problema, evitando duplicación de información y multicolinealidad.


**2. Features principales (core)**

Se seleccionaron como núcleo del modelo los siguientes indicadores:

- ema_60
- roc_60
- roc_30

Estos indicadores presentan:

- Mayor magnitud de IC OOS dentro del conjunto evaluado
- Presencia en todos los regímenes de mercado
- Bajo gap entre IS y OOS
- Consistencia de signo en todos los análisis

Desde el punto de vista estructural, capturan:

- Tendencia (EMA)
- Momentum (ROC)

Este conjunto concentra la mayor parte de la señal disponible.


**3. Features complementarias**

Se identificaron indicadores con señal moderada que pueden aportar información adicional:

- atr_norm_10
- stoch_k_20

Estos indicadores:

- Tienen IC menor, pero positivo
- Aportan información distinta (volatilidad y oscilación)
- No son esenciales, pero pueden mejorar marginalmente el modelo

Su uso es opcional y debe validarse en la etapa de modelado.


**4. Features descartadas**

Se descartaron los siguientes grupos de indicadores:

- MACD
- Volume ratios
- Bandas de Bollinger
- RSI
- MOM
- EMAs y ROCs adicionales

Motivos:

- Baja o nula señal fuera de muestra
- Alta correlación con los indicadores seleccionados
- Redundancia en la información

Esto confirma que muchos indicadores técnicos representan variaciones del mismo factor subyacente.


**5. Feature set final**

Se definen dos configuraciones posibles:

Versión mínima:

`ema_60`, `roc_60`, `roc_30`

Versión extendida:

`ema_60`, `roc_60`, `roc_30`, `atr_norm_10`, `stoch_k_20`


**6. Conclusión**

El problema presenta baja dimensionalidad efectiva. La señal está concentrada en pocos factores robustos, principalmente asociados a tendencia y momentum, con comportamiento de tipo mean-reversion.

El uso de múltiples indicadores no mejora la señal y genera redundancia. Por lo tanto, el modelo debe construirse sobre un conjunto reducido de features bien seleccionadas, priorizando calidad sobre cantidad.


# **13. Interpretación económica de los factores**

Hay 2 niveles:

**1) Clasificación por tipo de señal**

Asignamos cada feature a una categoría:

- Momentum
- Mean reversion
- Volatilidad

**2) Validación con datos**

Verificamos:
- Signo del IC
- Comportamiento esperado

## **13.1. Código**

In [ ]:
feature_types = {
    "ema_60": "trend",
    "roc_60": "momentum",
    "roc_30": "momentum",
    "stoch_k_20": "mean_reversion",
    "atr_norm_10": "volatility",
}

interpretation_rows = []

for f in feature_types:
    for target in ["delta_60", "delta_90"]:
        ic = compute_ic_by_day(
            mnq_intraday_with_indicators,
            f,
            target
        )

        interpretation_rows.append({
            "feature": f,
            "type": feature_types[f],
            "target": target,
            "IC": ic,
            "sign": "positive" if ic > 0 else "negative"
        })

interpretation_df = pd.DataFrame(interpretation_rows)

interpretation_df

,feature,type,target,IC,sign
0,ema_60,trend,delta_60,-0.156550,negative
1,ema_60,trend,delta_90,-0.194155,negative
2,roc_60,momentum,delta_60,-0.144045,negative
3,roc_60,momentum,delta_90,-0.182031,negative
4,roc_30,momentum,delta_60,-0.110168,negative
5,roc_30,momentum,delta_90,-0.139204,negative
6,stoch_k_20,mean_reversion,delta_60,-0.071651,negative
7,stoch_k_20,mean_reversion,delta_90,-0.089873,negative
8,atr_norm_10,volatility,delta_60,0.090375,positive
9,atr_norm_10,volatility,delta_90,0.102836,positive


## **13.2. Interpretación de resultados**


**1. Qué significa el signo del IC**

* **IC negativo** → cuando el indicador sube, el precio tiende a bajar
* **IC positivo** → cuando el indicador sube, el precio tiende a subir

---

**2. Interpretación por tipo de indicador**

- Trend / Momentum (EMA, ROC)

  Resultados:

  * ema_60 → negativo
  * roc_60 → negativo
  * roc_30 → negativo

  Interpretación:

  * Cuando el precio viene subiendo (momentum alto)
  * o está por encima de su tendencia

  > **tiende a revertir y bajar**

  Conclusión:

  * El mercado es **mean-reverting**, no tendencial

---

- Mean reversion (STOCH)

  Resultados:

  * stoch_k_20 → negativo

  Interpretación:

  * Cuando está en sobrecompra → cae
  * Cuando está en sobreventa → sube

  Conclusión:

  * Se comporta exactamente como debería
  * Confirma la lógica de reversión

---

- Volatilidad (ATR)

  Resultados:

  * atr_norm_10 → positivo

  Interpretación:

  * Cuando la volatilidad sube → los movimientos futuros son mayores
  * No indica dirección, pero sí intensidad

  Conclusión:

  * Aporta información útil, pero distinta (no direccional)

---

**3. Lectura global (la más importante)**

  Todos los indicadores direccionales:

  * EMA
  * ROC
  * STOCH

  tienen **IC negativo**

---

**4. Conclusión clave**

El mercado que estás modelando es:

  **mean-reverting (reversión a la media)**

No es un mercado tendencial.

---

**5. Traducción práctica**

* Subidas recientes → probabilidad de caída
* Bajadas recientes → probabilidad de rebote

---

**6. Por qué esto es importante**

Porque define:

* cómo aprende el modelo
* cómo interpretar predicciones
* cómo diseñar estrategias después

---

**Conclusión final del punto 13**

* Tus features tienen sentido económico
* El comportamiento es coherente
* La señal está alineada con un mercado intradía mean-reverting

- Esto valida completamente el análisis anterior
(no es ruido, es estructura real)


# **14. Análisis de features de régimen**

## **14.1. Marco teórico**

Las variables de **régimen sí pueden aportar valor real**.

---

**1. Diferencia clave con `day_of_week`**

* `day_of_week` → constante dentro del día → inútil para IC intradía, eliminalos los flags de día.

* `regime` → cambia dentro del día → **sí tiene variabilidad**

Por eso **sí pueden influir en el target**
Los resultados ya mostraron que:

* el comportamiento cambia por régimen
* el IC cambia por régimen

Entonces: **el régimen explica parte de la dinámica del mercado**

---

**2. Decisión importante**

Decidimos entrenar un solo modelo porque sería más preciso y menos complejo. Además, es lo más simple y más robusto.

Elegimos un modelo único, por lo que el régimen debe entrar como feature

---

**3. Cómo incluirlo correctamente**

- No usaremos 5 flags (mal) por los siguientes problemas:

  * redundancia
  * colinealidad
  * ruido innecesario


Y lo recomendado para modelos es incluir una versión númerica:

```python
regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_targets["regime_id"] = (
    mnq_intraday_targets[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)
```

---

**4. ¿Aporta señal realmente?**

Aunque no midamos IC directamente, anteriormente ya lo validamos indirectamente:

  * IC cambia por régimen
  * ranking cambia por régimen
  * robustez cambia por régimen

- Eso prueba que el régimen contiene información

---

**5. Interpretación correcta**

El régimen le dice al modelo:

* contexto de liquidez
* volatilidad
* comportamiento estructural

Ejemplo:

* opening → más volatilidad
* overnight → más reversión
* regular → más estabilidad

---

**6. Decisión final**

* Mantenemos régimen como feature
* Convertirmos a una sola columna (`regime_id`)
* No usamos múltiples flags

---

**Conclusión simple**

* A diferencia de los días, **sí aporta señal**
* Es un feature **contextual importante**
* Debe estar en el modelo si no segmentas por régimen

## **14.2. Código**

In [ ]:
regime_cols = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
]

regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_targets["regime_id"] = (
    mnq_intraday_targets[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)

In [ ]:
mnq_intraday_targets.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day',
       'is_premarket', 'is_opening', 'is_regular', 'is_closing',
       'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
       'delta_60', 'delta_90', 'regime_id'],
      dtype='object')

In [ ]:
'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri','regime',

('is_premarket',
 'is_opening',
 'is_regular',
 'is_closing',
 'is_overnight',
 'is_mon',
 'is_tue',
 'is_wed',
 'is_thu',
 'is_fri',
 'regime')

In [ ]:
cols_to_drop = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
    "is_mon",
    "is_tue",
    "is_wed",
    "is_thu",
    "is_fri",
    "regime",
]

# eliminar solo las que existan (evita errores)
mnq_intraday = mnq_intraday_targets.drop(
    columns=[c for c in cols_to_drop if c in mnq_intraday_targets.columns]
)

mnq_intraday.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day',
       'delta_60', 'delta_90', 'regime_id'],
      dtype='object')

# **15. Tratamiento de variables OHLCV**

**1. Punto de partida**

El dataset contiene variables OHLCV en bruto:

- open
- high
- low
- close
- volume

Sin embargo, los targets definidos (`delta_60` y `delta_90`) se construyen a partir del precio de cierre (`close`), y los indicadores técnicos seleccionados también se calculan principalmente a partir de la dinámica del precio y, en algunos casos, del volumen.

Por esta razón, antes de incluir OHLCV en el modelo, es necesario justificar si realmente aportan información adicional.


**2. Problema con OHLC en bruto**

Las variables `open`, `high`, `low` y `close` en valores absolutos no representan una señal directamente utilizable por el modelo, por las siguientes razones:

- dependen del nivel de precio
- no son estacionarias
- pueden inducir al modelo a aprender escalas de precio en lugar de patrones
- son altamente redundantes con los indicadores técnicos ya construidos

En particular, `close` merece atención especial, ya que:

- los targets se calculan directamente a partir de `close`
- muchos indicadores seleccionados también derivan de `close`

Por lo tanto, incluir `close` en bruto como feature puede llevar al modelo a capturar relaciones triviales o poco generalizables.


**3. Qué información de OHLC ya está representada**

La información contenida en OHLC ya fue transformada mediante indicadores técnicos, por ejemplo:

- tendencia: `ema_60`
- momentum: `roc_60`, `roc_30`
- oscilación / reversión: `stoch_k_20`
- volatilidad: `atr_norm_10`

Esto implica que el modelo ya dispone de representaciones más informativas y más estables que el precio crudo.


**4. Caso particular de volume**

La variable `volume` también debe analizarse con cuidado.

En principio, el volumen puede aportar información útil, pero en este trabajo no se utiliza en bruto, sino transformado mediante ratios y variables derivadas. Esto es preferible porque:

- reduce dependencia de escala
- mejora comparabilidad entre observaciones
- extrae información relativa, más útil para modelado

Por lo tanto, no se justifica mantener `volume` en bruto si ya existen transformaciones derivadas más adecuadas.


**5. Criterio de decisión**

Se adopta el siguiente criterio:

- eliminar variables OHLC en bruto
- eliminar `volume` en bruto
- conservar únicamente transformaciones e indicadores derivados que ya resumen esa información de forma más útil

Esto permite:

- reducir redundancia
- evitar multicolinealidad
- evitar que el modelo aprenda el nivel de precio en lugar de la estructura del mercado
- trabajar con features más robustas e interpretables


**6. Conclusión**

Las variables OHLCV en bruto no se incorporan al modelo final porque su información ya está representada por indicadores técnicos seleccionados y, en el caso del volumen, por transformaciones más adecuadas.

En consecuencia, el modelo se construirá sobre variables derivadas e indicadores, y no sobre precios absolutos.

## **15.1. Código**

In [ ]:
ohlcv_features = ["open", "high", "low", "close", "volume"]
targets = ["delta_60", "delta_90"]

rows = []

for feature in ohlcv_features:
    for target in targets:
        ic = compute_ic_by_day(
            mnq_intraday_with_indicators,
            feature,
            target
        )

        rows.append({
            "feature": feature,
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
        })

ohlcv_ic_df = (
    pd.DataFrame(rows)
    .sort_values(["target", "abs_IC"], ascending=[True, False])
    .reset_index(drop=True)
)

ohlcv_ic_df

,feature,target,IC,abs_IC
0,close,delta_60,-0.389163,0.389163
1,low,delta_60,-0.387566,0.387566
2,high,delta_60,-0.386490,0.386490
3,open,delta_60,-0.384810,0.384810
4,volume,delta_60,0.052525,0.052525
5,close,delta_90,-0.460708,0.460708
6,low,delta_90,-0.458979,0.458979
7,high,delta_90,-0.457304,0.457304
8,open,delta_90,-0.455397,0.455397
9,volume,delta_90,0.059621,0.059621


In [ ]:
selected_indicators = [
    "ema_60",
    "roc_60",
    "roc_30",
    "stoch_k_20",
    "atr_norm_10",
]

features_to_compare = ohlcv_features + selected_indicators
targets = ["delta_60", "delta_90"]

rows = []

for feature in features_to_compare:
    for target in targets:
        ic = compute_ic_by_day(
            mnq_intraday_with_indicators,
            feature,
            target
        )

        rows.append({
            "feature": feature,
            "feature_type": "OHLCV" if feature in ohlcv_features else "indicator",
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
        })

compare_ohlcv_vs_indicators_df = (
    pd.DataFrame(rows)
    .sort_values(["target", "abs_IC"], ascending=[True, False])
    .reset_index(drop=True)
)

compare_ohlcv_vs_indicators_df

,feature,feature_type,target,IC,abs_IC
0,close,OHLCV,delta_60,-0.389163,0.389163
1,low,OHLCV,delta_60,-0.387566,0.387566
2,high,OHLCV,delta_60,-0.386490,0.386490
3,open,OHLCV,delta_60,-0.384810,0.384810
4,ema_60,indicator,delta_60,-0.156550,0.156550
5,roc_60,indicator,delta_60,-0.144045,0.144045
6,roc_30,indicator,delta_60,-0.110168,0.110168
7,atr_norm_10,indicator,delta_60,0.090375,0.090375
8,stoch_k_20,indicator,delta_60,-0.071651,0.071651
9,volume,OHLCV,delta_60,0.052525,0.052525


In [ ]:
features_to_check = [
    "close",
    "ema_60",
    "roc_60",
    "roc_30",
    "stoch_k_20",
    "atr_norm_10",
]

corr_matrix = (
    mnq_intraday_with_indicators[features_to_check]
    .corr()
)

corr_matrix

,close,ema_60,roc_60,roc_30,stoch_k_20,atr_norm_10
close,1.000000,-0.001922,-0.002994,-0.002344,0.016407,-0.243939
ema_60,-0.001922,1.000000,0.866117,0.897518,0.612556,-0.063904
roc_60,-0.002994,0.866117,1.000000,0.713680,0.379997,-0.065685
roc_30,-0.002344,0.897518,0.713680,1.000000,0.537944,-0.045756
stoch_k_20,0.016407,0.612556,0.379997,0.537944,1.000000,-0.059569
atr_norm_10,-0.243939,-0.063904,-0.065685,-0.045756,-0.059569,1.000000


## **15.2. Conclusión sobre variables OHLCV**

El análisis realizado muestra que las variables OHLC presentan una alta correlación con los targets (`delta_60` y `delta_90`). Sin embargo, esta relación no responde a una señal predictiva real, sino a una dependencia directa derivada de la forma en que los targets están construidos a partir del precio de cierre.

Adicionalmente, se verificó que el precio de cierre (`close`) no presenta correlación significativa con los indicadores técnicos seleccionados, lo que indica que estos últimos capturan información transformada y estructural del mercado, distinta del nivel absoluto del precio.

En este contexto, incluir variables OHLC en bruto introduce:

- relaciones triviales con el target
- potencial sesgo en el modelo
- redundancia respecto a los indicadores técnicos
- falta de estacionariedad

Por su parte, el volumen en bruto tampoco aporta valor significativo frente a sus transformaciones derivadas ya utilizadas.

En consecuencia, se decide eliminar todas las variables OHLCV en bruto (`open`, `high`, `low`, `close`, `volume`) del dataset final, manteniendo únicamente indicadores técnicos y variables derivadas que representan de forma más robusta y útil la dinámica del mercado.

# **16. Análisis de la variable `minute_of_day`**

La variable `minute_of_day` representa la posición temporal de cada observación dentro de la jornada intradía, capturando la estructura temporal del mercado a lo largo del día.

A diferencia de los indicadores técnicos, que buscan capturar relaciones directas con el movimiento futuro del precio, esta variable tiene un carácter contextual. En particular, permite modelar patrones intradía bien conocidos, como:

- mayor volatilidad durante la apertura
- menor actividad en períodos intermedios
- posibles incrementos de volatilidad hacia el cierre

Por este motivo, su utilidad no necesariamente se refleja en una alta correlación directa con el target, sino en su capacidad para aportar información sobre el contexto en el que se generan los movimientos del mercado.

En consecuencia, el análisis de esta variable se abordará en dos niveles:

- evaluación de su relación directa con el target mediante IC
- análisis de comportamiento del target segmentado por intervalos de tiempo intradía

El objetivo es determinar si `minute_of_day` aporta valor como variable contextual dentro del modelo, aun cuando su señal predictiva directa pueda ser limitada.

## **16.1. Código**

In [ ]:
#MEDIR IC DIRECTO CON EL TARGET
rows = []

for target in ["delta_60", "delta_90"]:
    ic = compute_ic_by_day(
        mnq_intraday_with_indicators,
        "minute_of_day",
        target
    )

    rows.append({
        "feature": "minute_of_day",
        "target": target,
        "IC": ic,
        "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
    })

minute_ic_df = pd.DataFrame(rows)
minute_ic_df

,feature,target,IC,abs_IC
0,minute_of_day,delta_60,0.006239,0.006239
1,minute_of_day,delta_90,0.013156,0.013156


Luego, el paso 2 es analizar el comportamiento del target por tramos intradía. Para eso, primero creamos buckets:

In [ ]:
df = mnq_intraday_with_indicators.copy()

df["minute_bucket"] = pd.cut(
    df["minute_of_day"],
    bins=6
)

df[["minute_of_day", "minute_bucket"]].head()

,minute_of_day,minute_bucket
datetime,,
2020-01-02 05:59:00-05:00,359,"(358.399, 459.167]"
2020-01-02 06:00:00-05:00,360,"(358.399, 459.167]"
2020-01-02 06:01:00-05:00,361,"(358.399, 459.167]"
2020-01-02 06:02:00-05:00,362,"(358.399, 459.167]"
2020-01-02 06:03:00-05:00,363,"(358.399, 459.167]"


In [ ]:
bucket_analysis_60 = (
    df.groupby("minute_bucket")["delta_60"]
    .agg(
        mean_delta_60="mean",
        std_delta_60="std",
        n_obs="count",
    )
    .reset_index()
)

bucket_analysis_90 = (
    df.groupby("minute_bucket")["delta_90"]
    .agg(
        mean_delta_90="mean",
        std_delta_90="std",
        n_obs="count",
    )
    .reset_index()
)


In [ ]:
bucket_analysis_60

,minute_bucket,mean_delta_60,std_delta_60,n_obs
0,"(358.399, 459.167]",0.076555,33.162478,130795
1,"(459.167, 559.333]",0.085705,62.895036,129500
2,"(559.333, 659.5]",0.361108,74.202766,129500
3,"(659.5, 759.667]",0.851315,55.173091,129500
4,"(759.667, 859.833]",1.773546,58.545493,129500
5,"(859.833, 960.0]",-0.255156,59.632832,53095


In [ ]:
bucket_analysis_90

,minute_bucket,mean_delta_90,std_delta_90,n_obs
0,"(358.399, 459.167]",-0.325815,46.349664,130795
1,"(459.167, 559.333]",0.399178,81.698640,129500
2,"(559.333, 659.5]",0.438029,86.007831,129500
3,"(659.5, 759.667]",1.471583,69.515279,129500
4,"(759.667, 859.833]",2.191925,72.233943,129500
5,"(859.833, 960.0]",0.969937,74.375065,14245


## **16.2. Conclusiones sobre la variable minute_of_day**



El análisis realizado muestra que la variable `minute_of_day` no presenta necesariamente una alta correlación directa con los targets (`delta_60` y `delta_90`). Sin embargo, esto no implica que carezca de valor dentro del modelo.

Al analizar el comportamiento de los targets segmentado por intervalos intradía, se observa que:

- la media de los retornos cambia de forma significativa a lo largo del día
- la volatilidad (desviación estándar) también varía entre distintos tramos horarios
- existe una estructura clara en la evolución intradía del mercado

En particular, se identifican patrones como:

- valores más bajos o negativos en etapas tempranas
- incrementos progresivos en la media hacia períodos posteriores del día
- cambios relevantes en la dispersión de los retornos

Estos resultados evidencian que la dinámica del mercado no es homogénea durante la jornada, sino que depende del contexto temporal en el que se encuentra.

En este sentido, `minute_of_day` no actúa como un predictor directo de la dirección del precio, sino como una variable contextual que permite al modelo diferenciar entre distintos regímenes intradía implícitos.

Por lo tanto:

- no se espera que esta variable tenga una señal predictiva fuerte por sí sola
- pero sí aporta información estructural relevante
- complementa a los indicadores técnicos, que capturan la dinámica del precio

En consecuencia, se decide mantener `minute_of_day` dentro del conjunto final de features, ya que permite incorporar la estacionalidad intradía y mejorar la capacidad del modelo para adaptarse a distintos contextos de mercado.

# **17. Cierre del stage de selección de features**

**1. Features seleccionadas**

A partir del análisis de señal, robustez, consistencia y redundancia, se define el siguiente conjunto final de variables:

Features estructurales (señal directa):
- ema_60
- roc_60
- roc_30
- stoch_k_20
- atr_norm_10

Features contextuales:
- minute_of_day
- regime_id


**2. Features eliminadas**

Durante el proceso se descartaron las siguientes variables:

- Variables OHLC en bruto (open, high, low, close)
- Volumen en bruto (volume)
- Flags de día de la semana (is_mon, ..., is_fri)
- Flags de régimen individuales (reemplazados por regime_id)

Las razones principales fueron:

- relaciones triviales con el target
- redundancia con indicadores técnicos
- falta de estacionariedad
- ausencia de señal predictiva útil en el contexto intradía


**3. Estructura del feature set final**

El conjunto final combina:

- indicadores técnicos que capturan la dinámica del precio (tendencia, momentum, reversión, volatilidad)
- variables contextuales que permiten modelar la estructura del mercado (tiempo intradía y régimen)

Esta combinación permite al modelo:

- identificar patrones en los datos
- adaptarse a distintos contextos de mercado
- evitar dependencia de variables no informativas


**4. Validación del proceso**

El feature set final cumple con los criterios definidos:

- presenta señal out-of-sample
- muestra robustez entre IS y OOS
- mantiene consistencia de signo
- evita redundancia y multicolinealidad


**5. Conclusión**

Se obtiene un conjunto de variables compacto, informativo y coherente con la dinámica del mercado intradía.

El dataset resultante está preparado para las siguientes etapas del pipeline:

- partición temporal de datos
- construcción de ventanas
- entrenamiento de modelos predictivos

In [ ]:
mnq_intraday_targets.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day',
       'is_premarket', 'is_opening', 'is_regular', 'is_closing',
       'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
       'delta_60', 'delta_90', 'regime_id'],
      dtype='object')

In [ ]:
mnq_intraday_targets

,date,open,high,low,close,volume,minute_of_day,is_premarket,is_opening,is_regular,is_closing,is_overnight,is_mon,is_tue,is_wed,is_thu,is_fri,delta_60,delta_90,regime_id
datetime,,,,,,,,,,,,,,,,,,,,
2020-01-02 04:30:00-05:00,2020-01-02,8813.25,8813.25,8812.50,8813.25,13,270,0,0,0,0,1,0,0,0,1,0,6.00,4.75,0
2020-01-02 04:31:00-05:00,2020-01-02,8812.50,8812.50,8811.00,8812.50,121,271,0,0,0,0,1,0,0,0,1,0,6.25,5.25,0
2020-01-02 04:32:00-05:00,2020-01-02,8812.50,8813.25,8811.75,8811.75,53,272,0,0,0,0,1,0,0,0,1,0,7.25,4.75,0
2020-01-02 04:33:00-05:00,2020-01-02,8812.00,8812.25,8810.50,8810.50,37,273,0,0,0,0,1,0,0,0,1,0,7.75,7.00,0
2020-01-02 04:34:00-05:00,2020-01-02,8810.75,8812.25,8810.50,8812.00,36,274,0,0,0,0,1,0,0,0,1,0,6.75,6.25,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,0,0,0,1,0,0,0,0,0,1,NaN,NaN,4
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,0,0,0,1,0,0,0,0,0,1,NaN,NaN,4
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,0,0,0,1,0,0,0,0,0,1,NaN,NaN,4


In [ ]:
regime_cols = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
]

regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_targets["regime_id"] = (
    mnq_intraday_targets[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)

In [ ]:
regime_cols = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
]

regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_targets["regime_id"] = (
    mnq_intraday_targets[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)

cols_to_drop = [
    "open",
    "high",
    "low",
    "volume",
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
    "is_mon",
    "is_tue",
    "is_wed",
    "is_thu",
    "is_fri",
]

# eliminar solo las que existan (evita errores)
mnq_intraday = mnq_intraday_targets.drop(
    columns=[c for c in cols_to_drop if c in mnq_intraday_targets.columns]
)

mnq_intraday.columns

def calculte_technical_indicators(df=mnq_intraday_targets, target='close'):
    technical_indicators_columns = ['ema_60', 'roc_30','roc_60', 'stoch_k_20', 'atr_norm_10']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo

In [ ]:
mnq_intraday_features = calculte_technical_indicators(df=mnq_intraday_targets, target='close')

In [ ]:
mnq_intraday_features

(                                 date      open      high       low     close  \
 datetime                                                                        
 2020-01-02 04:30:00-05:00  2020-01-02   8813.25   8813.25   8812.50   8813.25   
 2020-01-02 04:31:00-05:00  2020-01-02   8812.50   8812.50   8811.00   8812.50   
 2020-01-02 04:32:00-05:00  2020-01-02   8812.50   8813.25   8811.75   8811.75   
 2020-01-02 04:33:00-05:00  2020-01-02   8812.00   8812.25   8810.50   8810.50   
 2020-01-02 04:34:00-05:00  2020-01-02   8810.75   8812.25   8810.50   8812.00   
 ...                               ...       ...       ...       ...       ...   
 2025-06-13 15:56:00-04:00  2025-06-13  21624.50  21635.00  21613.50  21617.50   
 2025-06-13 15:57:00-04:00  2025-06-13  21616.50  21635.25  21615.75  21623.75   
 2025-06-13 15:58:00-04:00  2025-06-13  21623.25  21632.75  21616.50  21621.75   
 2025-06-13 15:59:00-04:00  2025-06-13  21622.00  21632.25  21618.00  21628.00   
 2025-06-13 16:0